# Verificación de la simulación Metropolis con intercambio de réplicas en nanodiscos con DMI

**Sebastián Mindiola** · Computación Estadística 2026-II · Universidad Nacional de Colombia, sede La Paz

Este cuaderno es la etapa de **verificación** del proyecto sobre el muestreo de Metropolis con propuesta por transformada inversa en un nanodisco magnético con interacción Dzyaloshinskii–Moriya (DMI). Recibe la salida del cuaderno de simulación, que se ejecuta aparte en Kaggle: un archivo por estado de parámetros con las series de energía $H$, magnetización normalizada $m$ y carga topológica $Q$ de cada temperatura y de cada copia independiente, además de los diagnósticos del intercambio de réplicas (aceptaciones, viajes redondos) y de la termalización. Ningún resultado de este cuaderno se calcula con simulaciones nuevas: todo procede de esos archivos.

La verificación responde a una pregunta previa a la interpretación física: **si las cifras que entrega la simulación merecen confianza**. Con ese fin se revisan, en orden, los generadores de direcciones, el algoritmo completo frente a un resultado exacto, la dependencia entre muestras sucesivas y el tamaño de bloque del jackknife, la convergencia del intercambio de réplicas y, por último, la incertidumbre de $c(T)$, $\chi(T)$ y de la temperatura del pico $T^*$ mediante bootstrap.

| Bloque | Pregunta |
|---|---|
| Generadores | ¿Producen direcciones uniformes, en la esfera completa y en un casquete? |
| Algoritmo frente a Langevin | ¿El Metropolis (uniforme, en cono y con intercambio) reproduce un resultado exacto? |
| Autocorrelación y bloques | ¿Cuánto se correlacionan las muestras y qué bloque necesita el jackknife? |
| Intercambio y convergencia | ¿Las cadenas alcanzaron el equilibrio y la escalera mezcla? |
| Bandas de $c$ y $\chi$ y $T^*$ | ¿Qué tan inciertas son las curvas térmicas y la temperatura de transición? |

El orden de lectura sigue el de la tabla. Cada bloque abre con la teoría que lo justifica, continúa con tablas y figuras, y cierra con una guía de lectura que explica qué se hizo, por qué esa prueba es adecuada y qué se espera. La verificación se apoya en una idea: ninguna prueba demuestra que el algoritmo es correcto, pero cada una descarta una clase de error, de modo que su conjunto delimita en qué temperaturas y con qué reservas pueden usarse los resultados.


In [ ]:
# JAX con soporte CUDA: si la imagen de Kaggle no lo trae, lo instalo
import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import jax; print(jax.default_backend())"], capture_output=True, text=True)
print("backend actual:", r.stdout.strip() or r.stderr[-300:])
if r.stdout.strip() != "gpu":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[cuda12]"], check=True)

import os
# el proceso del cuaderno no reserva la memoria de la GPU por adelantado: la simulación corre en un subproceso
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import time, math, glob
import numpy as np
import matplotlib.pyplot as plt
import jax, jax.numpy as jnp
from jax import random, lax
from jax.scipy.special import gammaincc
from scipy import stats
from IPython.display import Markdown, display

print("JAX", jax.__version__, "| dispositivos:", jax.devices())
SEMILLA, ALPHA = 42, 0.05
os.makedirs("figures", exist_ok=True); os.makedirs("tablas", exist_ok=True)
t0_global = time.time()
# Región de la transición orden–desorden: allí se busca T* y se evalúa la convergencia; por debajo no hay equilibrio.
VENTANA_T = (6.0, 20.0)   # K


In [ ]:
import glob, shutil

# En Kaggle los resultados del cuaderno de simulacion llegan como dataset de entrada; en local se usan los que
# esten en el directorio de trabajo (fallback para poder generar y revisar el cuaderno sin volver a correr nada).
if os.path.isdir("/kaggle/input"):
    for raiz, dirs, archivos in os.walk("/kaggle/input"):
        if raiz.count(os.sep) <= 5:
            print("entrada:", raiz, dirs[:6], archivos[:6])
RUTAS = glob.glob("/kaggle/input/**/resultados_rd18p5", recursive=True)
DATOS = RUTAS[0] if RUTAS else "resultados_rd18p5"
print("resultados en:", DATOS, "| existe:", os.path.isdir(DATOS))
assert os.path.isdir(DATOS), "no se encontraron los resultados de la simulación en /kaggle/input"

for ruta in glob.glob("/kaggle/input/**/sim_pt.py", recursive=True):
    if not os.path.exists("sim_pt.py"):
        shutil.copy(ruta, "sim_pt.py"); print("simulador copiado desde", ruta)
if os.path.exists("sim_pt.py"):
    import importlib, sim_pt as sim
    importlib.reload(sim)
    print("simulador importado; los parámetros efectivos de la corrida se leen de los npz, no de las constantes del módulo")
else:
    print("AVISO: no se encontró sim_pt.py; las verificaciones E1 y E2 lo necesitan")

def cargar(dir_datos):
    out = {}
    for f in sorted(glob.glob(f"{dir_datos}/PT_est*.npz")):
        d = dict(np.load(f))
        # los barridos de termalización se recalculan del historial de trozos para no depender del metadato del npz
        d["barridos_term"] = d["term_hist"].shape[0] * int(d["mcs_term"]) // 10
        out["A" if abs(float(d["kDM"]) - 1.0) < 1e-6 else "B"] = d
    return out

PRINCIPAL = cargar(DATOS)
for et, d in PRINCIPAL.items():
    print(f"estado {et}: {int(d['n_spins'])} espines | {int(d['nt'])} temperaturas | {int(d['copias'])} copias | "
          f"{d['H'].shape[0]} muestras | puerta {'cumple' if bool(d['puerta_ok']) else 'NO CUMPLE'} | "
          f"{int(d['barridos_term'])} barridos de termalización")


## Observables termodinámicos: energía, calor específico, susceptibilidad y descriptores de textura

Cada corrida entrega, para cada estado de parámetros, cada temperatura de la escalera y cada copia, series de la energía total $H$ (en meV), de la magnetización normalizada $m$, de la carga topológica $Q$ y, en las temperaturas más bajas, del perfil radial de $S_z$. Las propiedades térmicas del disco no son esas series sino funcionales de la distribución de Boltzmann $\pi_\beta(\mathbf s)=e^{-\beta H(\mathbf s)}/Z(\beta)$ que las series muestrean. Algunas son **medias** (la energía) y otras son **varianzas** (el calor específico y la susceptibilidad). Esa diferencia decide cuánta muestra hace falta y cómo se calcula el error.

### Energía normalizada

Se define $e=\langle H\rangle/E_0$ con $E_0=(6J+6k_{DM})\,N\,S_0^2\,(+H_{ex}NS_0)$, donde $N$ es el número **real** de espines del disco ($N=5425$ para $R_d=18.5$) y no el número de nodos de la malla cuadrada que lo contiene: los espines fuera del disco valen cero y no aportan energía. $E_0$ no es la energía del estado fundamental sino una escala de referencia. Cada enlace de intercambio se cuenta una vez, de modo que aporta a lo sumo $3J$ por espín, y la DMI actúa solo en los enlaces $x$ e $y$, con a lo sumo $2k_{DM}$ por espín; la escala $6(J+k_{DM})$ es entonces una cota holgada. Su función es volver adimensional a $H$ para comparar estados con distinto $k_{DM}$ y discos de distinto tamaño.

### Calor específico y fluctuación–disipación

El calor específico es la respuesta de la energía media a un cambio de temperatura, $C=\partial\langle H\rangle/\partial T$. La derivación parte de la función de partición $Z(\beta)=\int e^{-\beta H}\,d\Omega$, con $d\Omega$ la medida uniforme sobre $(S^2)^N$:

$$\langle H\rangle=-\frac{\partial\ln Z}{\partial\beta},\qquad \frac{\partial\langle H\rangle}{\partial\beta}=-\frac{Z''}{Z}+\Big(\frac{Z'}{Z}\Big)^2=-\big(\langle H^2\rangle-\langle H\rangle^2\big)=-\operatorname{Var}(H).$$

Como $\beta=1/(k_BT)$, se tiene $d\beta/dT=-1/(k_BT^2)$ y, por la regla de la cadena,

$$C=\frac{\partial\langle H\rangle}{\partial\beta}\frac{d\beta}{dT}=\frac{\operatorname{Var}(H)}{k_BT^2},\qquad c(T)=\frac{C}{Nk_B}=\frac{\operatorname{Var}(H)}{N\,k_B^2\,T^2}.$$

La segunda igualdad expresa $c$ **por espín y en unidades de $k_B$** (con $H$ en meV y $k_B$ en meV/K, $k_BT$ es una energía y $c$ queda adimensional). Esta es una relación de **fluctuación–disipación**: una respuesta (cuánto cambia la energía al calentar) se mide sin perturbar el sistema, a partir de las fluctuaciones espontáneas en equilibrio. El resultado exige que $H$ no dependa de $T$; en este modelo los parámetros $J$, $k_{DM}$ y $K$ son fijos y no hay energía cinética, de modo que $c$ es puramente configuracional.

### Susceptibilidad y el papel de la norma

La magnetización normalizada es $m=\lVert\sum_i\mathbf S_i\rVert/N\in[0,1]$. Si se aplicara un campo $h$ (en meV) en una dirección fija $\hat{\mathbf n}$, $H\to H-h\,\hat{\mathbf n}\cdot\mathbf M$ con $\mathbf M=\sum_i\mathbf S_i$, y el mismo argumento de derivación da $\partial\langle M_{\hat n}\rangle/\partial h=\operatorname{Var}(M_{\hat n})/(k_BT)$. Por espín, $\chi=N\operatorname{Var}(m)/(k_BT)$, en meV$^{-1}$.

La norma se usa porque con $H_{ex}=0$ no existe dirección privilegiada a priori: la anisotropía cúbica admite seis direcciones fáciles equivalentes y la DMI induce texturas, de modo que el vector $\mathbf M$ puede apuntar en direcciones distintas entre copias y a lo largo del tiempo, y sus componentes se promediarían a cero aun con el disco ordenado. $m$ mide cuán alineado está el disco, sea cual sea la dirección. Esta definición tiene un costo: $\operatorname{Var}(m)$ recoge sobre todo las fluctuaciones **longitudinales** (de la longitud de $\mathbf M$). En fase desordenada, con tres componentes gaussianas de varianza $\sigma^2$, $\langle m^2\rangle\propto3\sigma^2$ mientras que la varianza de la norma es $(3-8/\pi)\sigma^2\approx0.45\,\sigma^2$. Por ello $\chi$ es un indicador operativo: su **posición** en temperatura es informativa; su valor absoluto no coincide con el de una susceptibilidad medida con campo.

### Temperatura de transición $T^*$

Se define $T^*=\arg\max_T c(T)$, con $T$ restringida a la ventana 6–20 K (región de la transición orden–desorden). En un sistema infinito una transición de fase de segundo orden produce un calor específico con singularidad. En un disco finito ($N=5425$, apenas cinco capas) $Z(\beta)$ es una integral de una función suave sobre una variedad compacta, por lo que es analítica: $c(T)$ es finita y suave, el pico es **redondeado** y su posición depende del tamaño, y no existe una transición estricta. $T^*$ es entonces una convención operativa, no una constante física.

Un pico plano tiene una consecuencia estadística. Cerca del máximo, $c(T)\approx c_{\max}-\tfrac12\kappa(T-T^*)^2$ con $\kappa$ pequeño; un error $\varepsilon$ en el valor de $c$ desplaza el máximo del orden de $\sqrt{2\varepsilon/\kappa}$. Un argmax sobre una curva plana está mal condicionado y su incertidumbre es mucho mayor que la de $c$ en cada punto, lo que justifica estimarla por remuestreo y no por propagación.

### Chequeo de normalización: $c\to k_B$ a baja temperatura

Un espín clásico tiene dos grados de libertad (la dirección en $S^2$ equivale a dos coordenadas locales). A temperatura baja, alrededor de un mínimo de $H$, la energía es cuadrática en las $2N$ coordenadas transversales, $H\approx H_{\min}+\tfrac12\sum_{a,b}Q_{ab}u_au_b$. Cada modo cuadrático aporta $\tfrac12k_BT$ por equipartición, de modo que $\langle H\rangle\approx H_{\min}+Nk_BT$ y $c\to k_B$. Esta cota sirve como control de unidades y de normalización ($N$ real, $k_B$ en meV/K). Las desviaciones tienen causas identificables: un modo quártico aporta $\tfrac14k_BT$ en lugar de $\tfrac12k_BT$; un modo plano no aporta energía; las correcciones anarmónicas crecen con $T$; y una cadena sin equilibrar da un valor sin significado. En corridas anteriores con otro radio ($R_d=9$, $N=1265$) se observó $c\to1\,k_B$, lo cual se cita solo como antecedente.

### Descriptores de textura: $Q$ y perfil radial

La carga topológica de Berg–Lüscher divide cada plaqueta en dos triángulos y suma el área esférica con signo que subtienden sus tres espines,

$$\Omega(\mathbf p,\mathbf q,\mathbf r)=2\operatorname{atan2}\big(\mathbf p\cdot(\mathbf q\times\mathbf r),\,1+\mathbf p\cdot\mathbf q+\mathbf q\cdot\mathbf r+\mathbf r\cdot\mathbf p\big),\qquad Q=\frac1{4\pi}\sum_{\text{triángulos}}\Omega,$$

calculada por capa y promediada en las $L$ capas. Cuenta cuántas veces la textura envuelve la esfera: vale cero para un disco uniforme o una espiral y $\pm1$ para un skyrmion (un remolino localizado de espines con núcleo invertido). Solo se incluyen plaquetas con las cuatro esquinas dentro del disco, por lo que $Q$ no es exactamente entero. El perfil radial de $S_z$ promedia $S_z$ sobre las capas y sobre anillos de radio $r=\operatorname{round}(\text{distancia al centro})$. Sin campo, la polaridad de una textura puede invertirse entre muestras, así que un promedio temporal del perfil solo es interpretable si la textura no cambia de signo.

### Por qué $c$ y $\chi$ son varianzas y por qué eso complica el error

El estimador $\hat c$ es una función no lineal de dos medias muestrales, $\overline{H^2}$ y $\bar H^2$. Tiene sesgo de orden $\tau/n$ y su error estándar no sigue la fórmula de una media: para datos gaussianos independientes, $\operatorname{Var}(s^2)=2\sigma^4/n_{\mathrm{ef}}$, por lo que el error relativo de una varianza es $\sqrt{2/n_{\mathrm{ef}}}$, mucho mayor que el de la media con la misma muestra. Además depende de la curtosis y de la autocorrelación de $(H-\bar H)^2$, que cerca de la transición decae más lento que la de $H$. Por eso la incertidumbre de $c$ y $\chi$ se obtiene con jackknife por bloques y bootstrap entre copias, descritos a continuación.


## Estadística de la incertidumbre con datos dependientes

Las muestras de una cadena de Metropolis no son independientes: cada configuración se obtiene modificando la anterior. Tratarlas como independientes produce errores estándar demasiado pequeños y, por tanto, conclusiones demasiado seguras. Esta sección describe cómo se mide la dependencia, cómo se corrige el error dentro de una cadena (jackknife por bloques) y cómo se combina la información entre cadenas independientes (bootstrap entre copias).

### Autocorrelación y tiempo integrado

Para una serie estacionaria $A_1,\dots,A_n$ con media $\mu$ y varianza $\sigma^2$, la autocorrelación a retardo $t$ es $\rho(t)=\operatorname{Cov}(A_s,A_{s+t})/\sigma^2$. La varianza de la media muestral es

$$\operatorname{Var}(\bar A)=\frac{\sigma^2}{n}\Big[1+2\sum_{t=1}^{n-1}\Big(1-\frac tn\Big)\rho(t)\Big]\approx\frac{\sigma^2}{n}\,\tau_{\mathrm{int}},\qquad \tau_{\mathrm{int}}=1+2\sum_{t=1}^{W}\rho(t),\qquad n_{\mathrm{ef}}=\frac{n}{\tau_{\mathrm{int}}}.$$

La fórmula de muestras independientes, $\sigma^2/n$, subestima el error en un factor $\sqrt{\tau_{\mathrm{int}}}$: la cadena equivale a $n_{\mathrm{ef}}$ muestras independientes y no a $n$. El tiempo $\tau_{\mathrm{int}}$ se expresa en unidades de muestras guardadas (una cada 2 barridos).

**Cálculo por FFT.** Se resta la media, se rellena la serie con ceros hasta una longitud al menos $2n$ (para que la correlación circular no mezcle los extremos), se calcula la transformada de Fourier, se toma el módulo al cuadrado y se aplica la transformada inversa. El resultado es $\sum_s x_sx_{s+t}$ para todos los retardos en $O(n\log n)$, en lugar de $O(n^2)$; se divide por el número de términos ($n-t$) o por $n$ según la convención, y por el valor en $t=0$. Esto importa porque hay una serie por cada estado, temperatura, copia y observable.

**Ventana de truncamiento.** Para $t$ grande, $\hat\rho(t)$ es casi puro ruido, y sumar todos los retardos añade ruido sin información. La ventana automática de Sokal elige el menor $W$ con $W\ge c\,\tau_{\mathrm{int}}(W)$, con $c=10$ en la convención del cuaderno, en la cual $\tau_{\mathrm{int}}=1+2\sum\rho$ es el doble de la $\tau$ de Sokal (1997). Por esa razón $c=10$ sobre esta $\tau_{\mathrm{int}}$ equivale a $c=20$ en la convención de Sokal: una ventana más conservadora que el valor habitual $c\approx4$–$6$, que reduce el sesgo por truncamiento a costa de algo más de varianza. Equilibra el sesgo por truncar demasiado pronto contra la varianza por truncar tarde.

### Jackknife por bloques

La serie se parte en $g$ bloques contiguos de largo $b$, con $n=gb$. Si $\hat\vartheta$ es el estimador con todos los datos y $\hat\vartheta_{(j)}$ el estimador sin el bloque $j$,

$$\widehat{\mathrm{se}}_{JK}=\Big[\frac{g-1}{g}\sum_{j=1}^g(\hat\vartheta_{(j)}-\bar\vartheta_{(\cdot)})^2\Big]^{1/2},\qquad \widehat{\mathrm{bias}}_{JK}=(g-1)(\bar\vartheta_{(\cdot)}-\hat\vartheta),$$

con $\bar\vartheta_{(\cdot)}=g^{-1}\sum_j\hat\vartheta_{(j)}$; el estimador corregido por sesgo es $g\hat\vartheta-(g-1)\bar\vartheta_{(\cdot)}$. Para la media, $\hat\vartheta_{(j)}-\bar\vartheta_{(\cdot)}=-(\bar A_j-\bar A)/(g-1)$ y se obtiene $\widehat{\mathrm{se}}^2=\sum_j(\bar A_j-\bar A)^2/[g(g-1)]$: la varianza de las medias de bloque dividida por $g$, que para bloques casi independientes vale $\approx\tau_{\mathrm{int}}\sigma^2/n$. Se eliminan bloques y no observaciones sueltas porque quitar una sola muestra conserva a sus vecinas correlacionadas y reproduce el error ingenuo. Para $c$ y $\chi$, cada réplica recalcula la media y la varianza con la serie restante.

**Regla del tamaño de bloque.** Dos sesgos actúan en sentido opuesto:

- Con $b$ pequeño frente a $\tau_{\mathrm{int}}$, los bloques siguen correlacionados entre sí y el error resulta subestimado. El sesgo relativo de la varianza es $\approx\tau_{\mathrm{int}}/(2b)$, que queda en $\le5\,\%$ si $b\ge10\,\tau_{\mathrm{int}}$.
- Con $b$ grande quedan pocos bloques y el error estimado es ruidoso. Su error relativo es $\approx1/\sqrt{2(g-1)}$, es decir, $\approx16\,\%$ con $g=20$.

Se adopta $b^*$ como el menor tamaño disponible con $b\ge10\tau$ y $g=n/b\ge20$. Con $n=8000$ muestras por copia, ambas condiciones son compatibles solo si $\tau\le n/200=40$ muestras; en caso contrario la temperatura se marca **no confiable**. Como $c$ y $\chi$ dependen de $\bar H$ y $\overline{H^2}$ (y de $m$ de forma análoga), su fluctuación está dominada por la serie más lenta; por ello $\tau$ es el máximo entre $H$, $m$, $(H-\bar H)^2$ y $(m-\bar m)^2$.

**Verificación por meseta.** Al graficar $\widehat{\mathrm{se}}_{JK}(b)$ frente a $b$, el error parte del valor ingenuo, crece mientras $b$ no supera a $\tau$ y se estabiliza cuando los bloques son efectivamente independientes; para los estimadores tipo media, la meseta debe coincidir con $\sqrt{\tau\hat\sigma^2/n}$. Si no hay meseta, la serie es corta o $\tau$ está subestimado.

### Bootstrap entre copias

El bootstrap clásico exige unidades independientes. Remuestrear instantes dentro de una cadena destruye la dependencia temporal y devuelve el error ingenuo. Las **copias** (escaleras completas con arranques y números aleatorios independientes) sí son independientes, de modo que se remuestrean copias enteras con reemplazo. Los índices se generan por inversión discreta: con $U\sim\mathcal U[0,1)$, $I=\lfloor RU\rfloor$ es uniforme en $\{0,\dots,R-1\}$ porque $P(I=i)=P(i/R\le U<(i+1)/R)=1/R$ (conviene acotar $I\le R-1$ por redondeo en FP32). Con $B=2000$ remuestras se recalcula el estimador agrupando las series de las copias elegidas.

**Intervalo percentil.** Los cuantiles 2.5 % y 97.5 % de las réplicas $\hat\vartheta^*_\ell$.

**Intervalo BCa.** Corrige el sesgo mediano y la asimetría. Con $\Phi$ la distribución normal estándar,

$$z_0=\Phi^{-1}\!\Big(\frac{\#\{\hat\vartheta^*_\ell<\hat\vartheta\}}{B}\Big),\qquad a=\frac{\sum_i(\bar\vartheta_{(\cdot)}-\hat\vartheta_{(i)})^3}{6\big[\sum_i(\bar\vartheta_{(\cdot)}-\hat\vartheta_{(i)})^2\big]^{3/2}},$$

$$\alpha_k=\Phi\Big(z_0+\frac{z_0+z^{(k)}}{1-a(z_0+z^{(k)})}\Big),\qquad z^{(1)}=\Phi^{-1}(0.025),\ z^{(2)}=\Phi^{-1}(0.975),$$

y el intervalo toma los cuantiles $\alpha_1$ y $\alpha_2$ de las réplicas. El factor de aceleración $a$ se calcula con las réplicas jackknife de la **muestra original** (se deja fuera una copia a la vez) y no con el bootstrap. La inversa normal se obtiene por bisección sobre $\Phi(z)=\tfrac12[1+\operatorname{erf}(z/\sqrt2)]$, que es monótona. Si $z_0=0$ y $a=0$, BCa coincide con el percentil; difieren cuando la distribución bootstrap está sesgada o es asimétrica, como suele ocurrir con varianzas cerca del pico. Si la proporción de réplicas por debajo de $\hat\vartheta$ es 0 o 1, $z_0$ no está definida y $B$ resulta insuficiente.

### Repesado de histogramas y tamaño efectivo

Las muestras de la temperatura $T_t$ informan sobre una temperatura vecina porque $\pi_\beta/\pi_{\beta_t}\propto e^{-(\beta-\beta_t)H}$. Por muestreo de importancia autonormalizado (Ferrenberg–Swendsen, 1988),

$$\langle A\rangle_\beta=\frac{\sum_nA_n\,w_n}{\sum_nw_n},\qquad w_n=e^{-(\beta-\beta_t)H_n},$$

con exponentes restados de su máximo para evitar desbordes. De los momentos ponderados de $H$ y $m$ se obtienen $c$ y $\chi$ en $\beta$. El **tamaño efectivo de Kish**, $n_K=(\sum w)^2/\sum w^2$, mide cuántas muestras equivalen a los pesos; se exige $n_K\ge10\,\%$ de las muestras para que unas pocas muestras de la cola no dominen. Cada punto de la rejilla se calcula desde la ranura más cercana en $\beta$.

### $T^*$ sobre rejilla fina

La escalera geométrica entre 0.5 y 20 K con 256 temperaturas tiene un paso relativo de $\approx1.5\,\%$, grueso frente a un pico plano, y el argmax sobre ella tomaría pocos valores discretos. Por eso $T^*$ se busca en una rejilla de 0.01 K sobre la curva repesada, dentro de la ventana 6–20 K. Para su incertidumbre, en cada remuestra de copias se recalcula la curva (con sumas ponderadas por copia) y se registra su argmax. Un máximo en el borde de la ventana indica que el pico está fuera de ella. Como el argmax no es una función suave de los datos, el jackknife que alimenta el factor $a$ puede ser inconsistente para este estimador y el intervalo percentil es más transparente.

### Limitaciones

- **FP32.** Las series se guardan en precisión simple. Calcular $\overline{H^2}-\bar H^2$ en FP32 cancela dígitos catastróficamente cuando $\operatorname{Var}(H)\ll\bar H^2$ (baja temperatura); conviene centrar en doble precisión.
- **Número de copias.** Con $R=16$, el bootstrap tiene solo 16 unidades; el percentil tiende a ser algo estrecho (la varianza bootstrap de una media es $(R-1)/R$ de la muestral) y los cuantiles extremos de BCa son frágiles.
- **Repesado lejos de $T_t$.** Fuera del traslape de las distribuciones de energía de ranuras vecinas, los pesos se concentran y el resultado pierde fiabilidad; el criterio de Kish lo detecta.
- **Equilibrio.** El bootstrap cuantifica la variabilidad entre copias, no el sesgo por falta de equilibrio; los intervalos de una temperatura marcada como provisional no certifican el valor de equilibrio.


## Criterios de convergencia y equilibrio

Todo el análisis de incertidumbre presupone que las cadenas muestrean la distribución de Boltzmann $\pi_\beta$. Un error estándar pequeño no garantiza eso: una cadena atrapada en un estado metaestable (una textura de energía más alta de la que domina el equilibrio) puede dar medias muy estables y erróneas. Esta sección describe los criterios que se aplican para detectarlo, la ventana de temperaturas en que se evalúan y cómo se combinan.

### Qué significa equilibrio

Una cadena de Markov irreducible y aperiódica con distribución invariante $\pi_\beta$ converge en ley a $\pi_\beta$ desde cualquier configuración inicial, pero no indica en qué momento. Operativamente, una cadena está en equilibrio cuando sus promedios (i) no dependen del punto de partida y (ii) no muestran deriva en el tiempo. La convergencia no se demuestra con los datos; solo se pueden detectar fallas. Por eso cada temperatura se arranca desde condiciones **sobredispersas**: 8 copias desordenadas (alta energía) y 8 ordenadas en $+z$. Si todas coinciden tras la termalización, es poco probable que compartan un mismo atasco. Todos los criterios son necesarios pero no suficientes: ninguno prueba equilibrio, y la ausencia de alarmas es evidencia parcial.

### Ventana de evaluación y reporte del total

Los criterios se evalúan en la ventana $6\le T\le20$ K, que contiene el pico del calor específico, y se reportan por separado sobre toda la escalera (0.5–20 K). La razón es estadística: con el presupuesto de la corrida, debajo de aproximadamente 2 K (estado A) y 6 K (estado B) la cadena no equilibra, y mezclar esas temperaturas con las demás haría que el veredicto dependiera de una región donde no se pretende concluir. Se muestran ambos resultados para que la restricción sea transparente: el veredicto de convergencia rige para la ventana, y el total documenta cuánto de la escalera queda fuera de ella.

### $\hat R$ dividido de Gelman–Rubin

La energía de cada copia se parte en dos mitades, lo que da $M=2R$ cadenas de longitud $n$ (la mitad de las muestras). Con $\bar\theta_k$ y $s_k^2$ la media y la varianza muestral (con $n-1$) de la cadena $k$, y $\bar{\bar\theta}$ la media global:

$$B=\frac{n}{M-1}\sum_{k=1}^M(\bar\theta_k-\bar{\bar\theta})^2,\quad W=\frac1M\sum_{k=1}^Ms_k^2,\quad \widehat{\mathrm{var}}^+=\frac{n-1}{n}W+\frac Bn,\quad \hat R=\sqrt{\frac{\widehat{\mathrm{var}}^+}{W}}.$$

$W$ es la variación dentro de cada cadena y $B$ la variación entre sus medias. Si todas muestrean la misma distribución estacionaria, $B/n$ y $W$ son estimaciones de la misma cantidad y $\hat R\approx1$; si las cadenas aún se acercan a su régimen estacionario, la varianza total supera a $W$ y $\hat R>1$. Partir cada copia en dos mitades permite detectar **deriva** dentro de una sola cadena: con una tendencia, las dos mitades tienen medias distintas aun cuando todas las copias se parezcan.

*Qué significa $\hat R\ge1.01$ para un estadístico.* El cociente $\hat R^2-1\approx B/(nW)$ mide qué fracción de la varianza total se debe a diferencias entre cadenas y no a ruido dentro de ellas. Un valor $\hat R=1.01$ equivale a que la varianza entre medias de cadena representa cerca del 2 % de $W$ (porque $\hat R^2\approx1.02$): las medias de las cadenas difieren más de lo que el ruido intra-cadena permite explicar, y las cadenas no son réplicas intercambiables de una misma distribución. En consecuencia, los intervalos construidos con ellas no cuantifican solo el error de muestreo, sino también una discrepancia sistemática. El umbral $\hat R<1.01$ es el recomendado por Vehtari et al. (2021). Dos límites: $\hat R$ compara medias y varianzas pero no corrige por autocorrelación, y al calcularse sobre la energía puede pasar por alto atascos que no se reflejan en $H$.

### Prueba de Welch entre arranques con corrección de Holm

Para cada estado y temperatura se compara la energía media por copia de las copias ordenadas frente a las desordenadas. La unidad es la copia, que es independiente:

$$t=\frac{\bar x_o-\bar x_d}{\sqrt{s_o^2/n_o+s_d^2/n_d}},\qquad \nu=\frac{(s_o^2/n_o+s_d^2/n_d)^2}{(s_o^2/n_o)^2/(n_o-1)+(s_d^2/n_d)^2/(n_d-1)}.$$

Se usa Welch y no Student porque no se supone igualdad de varianzas: un grupo atascado suele tener otra dispersión. Una diferencia significativa indica memoria del arranque. Con una prueba por temperatura, al nivel de 5 % se esperarían, bajo la hipótesis nula, falsas alarmas del orden de $0.05\times m$ en $m$ pruebas. Esto se controla con la corrección de **Holm**: se ordenan los $m$ valores $p_{(1)}\le\dots\le p_{(m)}$ y se ajusta $\tilde p_{(k)}=\max_{j\le k}\min\{1,(m-j+1)p_{(j)}\}$. Controla la probabilidad de al menos una falsa alarma y es uniformemente más potente que Bonferroni sin suponer independencia, lo que importa porque las temperaturas están acopladas por el intercambio. Con 8 copias por grupo la potencia es modesta: $p>0.05$ no demuestra igualdad. La familia de Holm se define por estado; se reportan el número de temperaturas con diferencia significativa dentro de la ventana y en total.

### Aceptación del intercambio

La aceptación entre las ranuras $t$ y $t+1$ es $a_t=\min\{1,\exp[(\beta_t-\beta_{t+1})(H_t-H_{t+1})]\}$, que procede de la razón entre el producto de Boltzmann tras y antes del intercambio. Es alta cuando las distribuciones de energía de ranuras vecinas se traslapan; depende de $(\Delta T/T)\sqrt{Nc}$, por lo que tiende a caer cerca del pico de $c$. Si algún par tiene aceptación baja, la escalera queda cortada: las ranuras frías no reciben configuraciones calientes y conservan la memoria del arranque. El umbral es $a_t\ge0.1$ en todos los pares. En los datos, la aceptación del par $t$ es `sw_acc[t] / (sw_int[t] · COPIAS)`.

### Viajes redondos totales frente a la teoría

Cada caminante (configuración que se mueve por la escalera con el intercambio) realiza un **viaje redondo** al tocar la ranura más caliente después de haber tocado la más fría y volver a esta. Cada viaje equivale a una configuración que se enfría, se decorrelaciona a alta temperatura y regresa, de modo que el número de viajes aproxima el número de muestras independientes en frío. Los viajes de una copia son los de todos sus caminantes.

*Deducción breve de la teoría.* El esquema determinista par/impar produce un movimiento casi balístico: un caminante que acaba de ser aceptado continúa en la misma dirección en el intento siguiente. Con aceptación total ($r_t=0$ para todo $t$), cada caminante recorre la escalera de $N_T$ ranuras en $N_T$ intentos y el viaje redondo dura $2N_T$ intentos; como hay $N_T$ caminantes, la copia acumula $N_T/(2N_T)=1/2$ viajes por intento, que es lo que verifica la autoprueba. Cuando un par rechaza con probabilidad $r_t=1-a_t$, el caminante espera en promedio $1/(1-r_t)=1+r_t/(1-r_t)$ intentos para cruzarlo; el exceso $r_t/(1-r_t)$ se suma sobre los pares y reduce la tasa. Con la teoría del intercambio no reversible (Syed et al., 2022) la tasa global por intento es

$$\frac{1}{2\big(1+\sum_t r_t/(1-r_t)\big)},$$

y los viajes esperados de una copia son

$$V_{\mathrm{teo}}=\frac{\text{intentos}_{\mathrm{term}}+\text{intentos}_{\mathrm{med}}}{2\big(1+\sum_t r_t/(1-r_t)\big)},\qquad\text{intentos}_{\mathrm{term}}=\frac{\text{barridos}_{\mathrm{term}}}{2},\quad\text{intentos}_{\mathrm{med}}=\frac{16\,000}{2}.$$

Se cuentan los viajes de **toda** la corrida (termalización más medición): `viajes + viajes_term`. Contar solo la medición era un planteamiento incorrecto, porque la teoría se refiere a los intentos totales y la medición por sí sola ofrece pocos viajes por copia (del orden de unos pocos).

*Criterio.* Se exige la mediana entre copias de los viajes totales $\ge10$ y $\ge0.5\,V_{\mathrm{teo}}$. La **autoprueba** del simulador fuerza la aceptación de todos los intercambios y comprueba que el contador cuente $\approx$ intentos/2, lo que separa un error de contabilidad de una mezcla lenta real. Una cifra muy inferior a la predicción indica que la escalera mezcla peor de lo que sugiere la aceptación.

### Puerta de equilibrio

Durante la termalización, la simulación guarda en trozos de 1 600 barridos la energía media por (estado, temperatura, copia). Las medias de dos trozos se agrupan en una ventana de 3 200 barridos y, para cada par (estado, temperatura) con $T\ge6$ K, se calculan dos estadísticos $z$: la **deriva** (los dos últimos trozos frente a los dos anteriores) y la diferencia entre copias ordenadas y desordenadas sobre los dos últimos. Un par es «malo» si algún $|z|>3.5$. La termalización termina cuando en dos evaluaciones seguidas no más del 2 % de los pares es malo, y puede ampliarse hasta 32 000 barridos; si no se cumple, el resultado declara **NO CUMPLE** y las temperaturas se marcan provisionales. Con datos estacionarios la fracción esperada de fallos es cercana a 0.5 %, por lo que la tolerancia del 2 % absorbe el ruido y detecta derivas reales. El umbral 3.5 reconoce que con 8 copias por grupo la normal subestima las colas.

### La marca «convergió / provisional»

Una temperatura de la ventana 6–20 K se marca **convergió** solo si cumple a la vez: puerta de equilibrio cumplida; aceptación de intercambio $\ge0.1$ en los pares adyacentes; viajes redondos totales suficientes y consistentes con la teoría; $\hat R<1.01$; y $p>0.05$ en Welch–Holm. Si falla alguna, es **provisional**; las temperaturas fuera de la ventana se tratan como no convergidas. La conjunción es deliberadamente conservadora: cada criterio detecta una falla distinta. La marca limita las conclusiones: los intervalos de una temperatura provisional describen la variabilidad entre copias, no certifican el equilibrio; un $T^*$ cuyo pico cae en una zona provisional debe presentarse como tentativo; y la ausencia de alarmas no prueba convergencia.

### Qué se hizo y qué representa

Se definió un conjunto de cinco criterios complementarios (puerta, aceptación, viajes redondos, $\hat R$ y Welch–Holm), evaluados en la ventana 6–20 K y reportados también en total. Cada uno representa una pregunta distinta: si la termalización terminó, si la escalera permite recorrer todas las temperaturas, si las copias coinciden entre sí y consigo mismas, y si el arranque se olvidó. Es la forma correcta porque el equilibrio solo puede refutarse, nunca probarse, y es el conjunto de pruebas que más tipos de falla cubre con las herramientas de un curso de estadística.


## Validación de los generadores por bondad de ajuste

El algoritmo de Metropolis depende de dos generadores de direcciones obtenidos por transformada inversa (sección de direcciones uniformes): la **esfera completa**, con $z=2U_1-1$ y $\varphi=2\pi U_2$, y el **casquete** de semiángulo $\alpha$ alrededor del espín actual, con $z'=1-U_1(1-\cos\alpha)$ y $\psi=2\pi U_2$ en el marco local, seguido de una rotación. Si alguno de los dos fuera defectuoso, la propuesta dejaría de ser simétrica o uniforme, y el cociente de Hastings dejaría de ser $e^{-\beta\Delta H}$. La uniformidad se comprueba con una prueba de bondad de ajuste $\chi^2$ repetida con muchas semillas.

**Diseño de la prueba.** El dominio se parte en $k=8\times8=64$ celdas **de igual probabilidad**:

- en la esfera, 8 franjas de igual altura en $z$ por 8 sectores de igual ancho en $\varphi$;
- en el casquete ($\alpha=\pi/4$), 8 franjas de igual altura en $z'=\mathbf S\cdot\mathbf S'\in[\cos\alpha,1]$ por 8 sectores en $\psi$, con ejes $\mathbf S$ aleatorios.

Se usan franjas en $z$ y no en $\theta$ porque la densidad uniforme en $z$ es constante, de modo que celdas de igual tamaño en $(z,\varphi)$ tienen igual probabilidad. Las celdas equiprobables dan a todas el mismo conteo esperado $n/64$, lo que evita que una celda con esperado diminuto domine el estadístico y garantiza que ninguna tenga un esperado menor que 5, condición habitual para que la aproximación $\chi^2$ sea válida. En el casquete, $\psi$ se mide en un marco **distinto** al que usa el generador, para no validar el código consigo mismo; un cambio de origen de $\psi$ independiente de $\psi$ conserva la uniformidad.

$$X^2=\sum_{j=1}^{64}\frac{(O_j-n/64)^2}{n/64}\ \xrightarrow{d}\ \chi^2_{63}.$$

- **Grados de libertad:** $63=64-1$, porque no se estiman parámetros.
- **Nivel:** $\alpha=0.05$, con valor crítico $\chi^2_{63,0.95}=82.53$.
- **Tamaño:** $n=4\times10^6$ direcciones por semilla, con $62\,500$ esperadas por celda; la muestra es grande para detectar desviaciones pequeñas.

**Ley binomial del número de rechazos.** Una sola prueba a nivel 5 % rechaza por azar una de cada veinte veces, de modo que un solo rechazo no es evidencia de defecto. Por eso se repite con **100 semillas independientes**. Bajo la hipótesis nula, cada prueba rechaza con probabilidad 0.05 y el número de rechazos sigue $\mathrm{Binomial}(100,0.05)$, con media 5 y desviación $\sqrt{100\cdot0.05\cdot0.95}\approx2.2$. El generador se declara defectuoso si el número de rechazos alcanza el **umbral $k=12$**, para el cual $P(X\ge12)<0.01$ bajo la hipótesis nula: la probabilidad de una falsa alarma queda por debajo del 1 %. Además, bajo $H_0$ los 100 $p$-valores son $\mathcal U(0,1)$, lo que se contrasta con una prueba de Kolmogorov–Smirnov calculada de forma explícita. Esto detecta un generador que, sin rechazar más de lo esperado, produce $p$-valores sesgados.

**Momentos.** La uniformidad en la esfera implica $\mathbb E[\mathbf S]=\mathbf 0$ y $\mathbb E[S_a^2]=1/3$ para cada componente $a$. Se comprueban a menos de 3 errores estándar; con $\operatorname{Var}(S_a)=1/3$ y $\operatorname{Var}(S_a^2)=1/5-1/9=4/45$, los errores estándar son $\sqrt{1/(3n)}$ y $\sqrt{4/(45n)}$. Esta prueba es complementaria: el $\chi^2$ examina la forma completa de la distribución y los momentos examinan sus efectos promedio sobre las cantidades que usa la física.

**Generador gaussiano como control.** Se incluye un tercer generador construido de otra manera: tres variables normales independientes divididas por su norma. Por la simetría rotacional de la gaussiana esférica, la dirección resultante es uniforme en $S^2$ sin recurrir a la transformada inversa. Pasa por el mismo código de prueba y tiene una tasa de rechazo conocida (5 % por semilla). Si el control pasa y el generador evaluado falla, el defecto está en el generador; si ambos fallan, el defecto está en la prueba.

**Antecedente.** En una versión previa del cuaderno, dos pruebas marginales dieron $p=0.019$ en $\varphi$. Un valor así es compatible con el azar entre muchas pruebas; la repetición con 100 semillas y el umbral binomial permiten distinguir azar de defecto.


In [ ]:
K_LADO, N_UNIF, N_SEMILLAS = 8, 4_000_000, 100
GL = K_LADO * K_LADO - 1
ALFA_CAS = math.pi / 4

def esfera_inversa(key, n):
    return sim.esfera_uniforme(key, (n,))

def esfera_gauss(key, n):
    g = random.normal(key, (n, 3))
    return g / jnp.sqrt(jnp.sum(g * g, axis=-1, keepdims=True))

def celdas_esfera(key, gen):
    v = gen(key, N_UNIF)
    phi = jnp.mod(jnp.arctan2(v[:, 1], v[:, 0]), 2 * jnp.pi)
    iz = jnp.clip(jnp.floor((v[:, 2] + 1) / 2 * K_LADO).astype(jnp.int32), 0, K_LADO - 1)
    ip = jnp.clip(jnp.floor(phi / (2 * jnp.pi) * K_LADO).astype(jnp.int32), 0, K_LADO - 1)
    return jnp.bincount(iz * K_LADO + ip, length=K_LADO**2), v

def celdas_casquete(key, _gen=None):
    k1, k2 = random.split(key)
    S = esfera_inversa(k1, N_UNIF)
    Sp = sim.propuesta_cono(k2, S, math.cos(ALFA_CAS))
    z = jnp.clip(jnp.sum(S * Sp, -1), math.cos(ALFA_CAS), 1.0)
    # marco de medición distinto al del generador (eje auxiliar y, o z si S es casi paralelo a y)
    aux = jnp.where(jnp.abs(S[:, 1:2]) < 0.9, jnp.array([0.0, 1.0, 0.0]), jnp.array([0.0, 0.0, 1.0]))
    f1 = jnp.cross(aux, S); f1 = f1 / jnp.linalg.norm(f1, axis=-1, keepdims=True); f2 = jnp.cross(S, f1)
    psi = jnp.mod(jnp.arctan2(jnp.sum(Sp * f2, -1), jnp.sum(Sp * f1, -1)), 2 * jnp.pi)
    iz = jnp.clip(jnp.floor((z - math.cos(ALFA_CAS)) / (1 - math.cos(ALFA_CAS)) * K_LADO).astype(jnp.int32), 0, K_LADO - 1)
    ip = jnp.clip(jnp.floor(psi / (2 * jnp.pi) * K_LADO).astype(jnp.int32), 0, K_LADO - 1)
    return jnp.bincount(iz * K_LADO + ip, length=K_LADO**2), Sp

def ks_uniforme(u):
    u = np.sort(np.asarray(u, dtype=np.float64)); n = u.size; i = np.arange(1, n + 1)
    return max(np.max(i / n - u), np.max(u - (i - 1) / n))

def p_ks(d, n):
    x = (math.sqrt(n) + 0.12 + 0.11 / math.sqrt(n)) * d
    return float(min(1.0, max(0.0, 2 * sum((-1) ** (j - 1) * math.exp(-2 * j * j * x * x) for j in range(1, 101)))))

def binom_cola_superior(k, n, p):
    lp = lambda j: math.lgamma(n + 1) - math.lgamma(j + 1) - math.lgamma(n - j + 1) + j * math.log(p) + (n - j) * math.log(1 - p)
    return sum(math.exp(lp(j)) for j in range(k, n + 1))

K_UMBRAL = next(k for k in range(N_SEMILLAS + 1) if binom_cola_superior(k, N_SEMILLAS, ALPHA) < 0.01)
print("umbral de rechazos (P(X>=k)<0.01):", K_UMBRAL)

resultados_E1 = {}
for nombre, fn, gen in (("esfera, inversa", celdas_esfera, esfera_inversa), ("esfera, gaussiana", celdas_esfera, esfera_gauss),
                        ("casquete π/4, inversa", celdas_casquete, None)):
    f = jax.jit(lambda k, fn=fn, gen=gen: fn(k, gen))
    pv, m1, m2 = [], [], []
    for s in range(N_SEMILLAS):
        c, v = f(random.PRNGKey(s))
        c = np.asarray(c, dtype=np.float64); esp = N_UNIF / c.size
        pv.append(float(gammaincc(GL / 2, float(((c - esp) ** 2 / esp).sum()) / 2)))
        m1.append(np.asarray(jnp.mean(v, 0))); m2.append(np.asarray(jnp.mean(v * v, 0)))
    pv, m1, m2 = map(np.array, (pv, m1, m2))
    rech = int((pv < ALPHA).sum()); d = ks_uniforme(pv)
    resultados_E1[nombre] = dict(p=pv, rechazos=rech, p_binom=binom_cola_superior(rech, N_SEMILLAS, ALPHA), ks=p_ks(d, N_SEMILLAS),
                                 m1=m1.mean(0), m1_se=m1.std(0, ddof=1) / 10, m2=m2.mean(0), m2_se=m2.std(0, ddof=1) / 10)
    print(f"{nombre:22s} rechazos {rech}/100  P(X>={rech})={resultados_E1[nombre]['p_binom']:.3f}  KS p={p_ks(d, N_SEMILLAS):.3f}")

In [ ]:
filas = [f"| {k} | {r['rechazos']} | {r['p_binom']:.3f} | {r['ks']:.3f} | {r['m1'][0]:+.5f} ± {r['m1_se'][0]:.5f} | {r['m2'][2]:.5f} ± {r['m2_se'][2]:.5f} |"
         for k, r in resultados_E1.items()]
tabla_E1 = ("| Generador | Rechazos de 100 | P(X ≥ rechazos) | p de KS sobre los p-valores | E[S_x] | E[S_z²] |\n"
            "|---|---|---|---|---|---|\n" + "\n".join(filas))
open("tablas/E1.md", "w").write(tabla_E1); display(Markdown(tabla_E1))
print("En el casquete los ejes S son uniformes, así que el punto propuesto también es uniforme en la esfera: sus momentos también deben ser 0 y 1/3.")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for a, (k, r) in zip(ax, resultados_E1.items()):
    a.hist(r["p"], bins=10, range=(0, 1), edgecolor="k"); a.axhline(10, ls="--", c="gray", label="esperado bajo H0")
    a.set_title(k); a.set_xlabel("p-valor")
ax[0].set_ylabel("frecuencia"); ax[0].legend()
plt.tight_layout(); plt.savefig("figures/E1_pvalores.png", dpi=150); plt.show()

def cumple_E1(r):
    return (r["rechazos"] < K_UMBRAL and r["ks"] > ALPHA and np.all(np.abs(r["m1"]) < 3 * r["m1_se"])
            and np.all(np.abs(r["m2"] - 1 / 3) < 3 * r["m2_se"]))
for k in ("esfera, inversa", "casquete π/4, inversa"):
    print(f"Verificación E1 ({k}):", "se cumple" if cumple_E1(resultados_E1[k]) else "no se cumple")

### Guía de lectura: validación de los generadores

**Qué se hizo.** Se generaron direcciones con los dos generadores del algoritmo (esfera completa y casquete alrededor de un eje aleatorio) y con el generador gaussiano de control. Para cada uno se repitió, con 100 semillas independientes, una prueba $\chi^2$ de bondad de ajuste sobre 64 celdas de igual probabilidad, y se registraron el número de rechazos al nivel 0.05, la distribución de los 100 $p$-valores y los momentos $\mathbb E[\mathbf S]$ y $\mathbb E[S_a^2]$ con su error estándar.

**Por qué es la prueba adecuada.** La propuesta de Metropolis solo se cancela en el cociente de Hastings si es simétrica, y para el generador de casquete esto equivale a que la dirección propuesta sea uniforme en el casquete. La bondad de ajuste $\chi^2$ examina la distribución completa y no un resumen. Las celdas equiprobables se construyen con franjas iguales en $z$ y no en $\theta$, y su conteo esperado es igual en todas, así que ninguna domina el estadístico. Una sola prueba rechaza por azar el 5 % de las veces; por eso se repite 100 veces y se juzga el **número** de rechazos con la ley $\mathrm{Binomial}(100,0.05)$ y el umbral de falla $k=12$. La prueba de Kolmogorov–Smirnov sobre los $p$-valores detecta defectos que no elevan la tasa de rechazo pero alteran su distribución. Los momentos comprueban el comportamiento promedio. El generador gaussiano, construido por otra vía, separa un defecto de la prueba de uno del generador.

**Qué representa el resultado y qué se espera.** Cada fila resume qué tan compatible es un generador con la uniformidad. Si el generador es correcto, el número de rechazos debe quedar cerca de 5 y por debajo del umbral, los $p$-valores deben distribuirse de forma plana en $(0,1)$ y los momentos deben estar a menos de 3 errores estándar de sus valores teóricos. Un número de rechazos cercano al umbral es una señal de advertencia que justifica más semillas; uno claramente por encima, un defecto. Pasar la prueba no demuestra que el generador sea exacto: solo descarta desviaciones detectables con $4\times10^6$ direcciones por semilla, suficientes para las que afectarían de manera apreciable al Metropolis.


## Comprobación del algoritmo contra una solución exacta

Para el espín $i$ se propone $\mathbf S_i'$ y se acepta con probabilidad

$$a=\min\Big\{1,\frac{\pi_\beta(\mathbf s')\,q(\mathbf S_i\mid\mathbf S_i')}{\pi_\beta(\mathbf s)\,q(\mathbf S_i'\mid\mathbf S_i)}\Big\}=\min\{1,e^{-\beta\Delta H_i}\},$$

porque la propuesta $q$ es simétrica y la constante $Z$ se cancela. La ventana del cono se adapta durante la termalización y se **fija** en la medición, de modo que la cadena medida es de Markov homogénea con $\pi_\beta$ invariante. Que la teoría diga que la cadena es correcta no demuestra que el código lo sea: un error de signo en $\Delta H$, una propuesta no simétrica o un jacobiano omitido pueden pasar inadvertidos. La prueba busca un caso donde la respuesta correcta se conoce exactamente.

**El modelo exacto.** Un solo espín en un campo, $H=-hS_z$, tiene $\pi_\beta(\mathbf S)\propto e^{x\cos\theta}$ con $x=\beta h$. Con $z=\cos\theta$ y densidad $\propto e^{xz}$ en $[-1,1]$,

$$Z=\int_{-1}^1e^{xz}dz=\frac{2\sinh x}{x},\qquad\langle S_z\rangle=\frac{d\ln Z}{dx}=\coth x-\frac1x,$$

que es la función de Langevin. Es una prueba **decisiva** por tres razones: la solución es exacta, sin aproximaciones; el sistema es tan simple que no hay vecinos ni tablero de ajedrez que contaminen el resultado, así que cualquier discrepancia es del algoritmo; y es sensible a los errores típicos. Por ejemplo, proponer $\theta$ uniforme sin incluir el factor $\sin\theta$ en el cociente de aceptación muestrea la densidad $e^{x\cos\theta}$ en $\theta$ y no en $z$, lo que sesga $\langle S_z\rangle$ de forma visible.

**Qué se comprueba.** Cada caso se corre con 64 cadenas independientes:

1. **Propuesta uniforme**, con 6 valores de $\beta h$ entre 0.1 y 10: verifica la regla de aceptación y el generador de esfera completa en todo el rango, desde campo débil (casi desordenado) hasta campo fuerte (casi alineado).
2. **Propuesta en cono** con $\alpha=\pi/4$ y los mismos 6 valores: verifica que la propuesta en casquete es simétrica y no introduce sesgo, pese a que cada paso solo explora un vecindario.
3. **Escalera de 8 temperaturas con intercambio de réplicas**, usando la función de intercambio del simulador: verifica que el intercambio deja invariante el producto $\prod_t\pi_{\beta_t}$, de modo que cada ranura conserva su propia distribución.

**Estadístico.** En cada comparación, el estimador de $\langle S_z\rangle$ es la media entre cadenas del promedio temporal. Su error estándar se corrige por autocorrelación con $\tau_{\mathrm{int}}$, $\widehat{\mathrm{se}}=\sqrt{\tau_{\mathrm{int}}\hat\sigma^2/n}$ combinado entre las 64 cadenas independientes. Se calcula

$$z=\frac{\bar S_z-(\coth x-1/x)}{\widehat{\mathrm{se}}},\qquad p=2\,[1-\Phi(|z|)],$$

que bajo la hipótesis de que el algoritmo es exacto es aproximadamente normal estándar por el teorema central del límite. Son $6+6+8=20$ comparaciones, así que se aplica la corrección de **Holm** para controlar la probabilidad de al menos una falsa alarma (véase la sección de convergencia). El algoritmo pasa si ningún $p$ ajustado es menor que 0.05.

**Alcance.** Un resultado sin rechazos no demuestra igualdad: solo descarta sesgos mayores que unos pocos errores estándar. La prueba valida la regla de aceptación, las propuestas y el intercambio, pero no la implementación de la energía de la red ni la actualización por tablero de ajedrez, que requieren otras comprobaciones.


In [ ]:
def tau_int_sokal_np(x, c=10.0):
    # tau = 1 + 2 sum_{t=1}^W rho(t); ACF por FFT con relleno de ceros; W mínimo con W >= c tau(W). x: (n,) o (n, K)
    x = np.asarray(x, dtype=np.float64); uno = x.ndim == 1
    x = x[:, None] if uno else x
    n = x.shape[0]; xc = x - x.mean(0)
    f = np.fft.rfft(xc, n=2 * n, axis=0)
    acf = np.fft.irfft(f * np.conj(f), n=2 * n, axis=0)[:n]
    a0 = acf[0]; rho = acf / np.where(a0 > 0, a0, 1.0)
    cum = 1.0 + 2.0 * np.cumsum(rho[1:], axis=0)
    cond = np.arange(1, n)[:, None] >= c * cum
    idx = np.where(cond.any(0), cond.argmax(0), n - 2)
    tau = np.maximum(cum[idx, np.arange(x.shape[1])], 1.0)
    return tau[0] if uno else tau

N_CAD, N_PASOS, N_CAL = 64, 100_000, 10_000
BH = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

def cadena(key, bh, cos_a):
    def paso(s, k):
        k1, k2 = random.split(k)
        sp = sim.propuesta_cono(k1, s, cos_a)
        acepta = random.uniform(k2) < jnp.exp(bh * (sp[2] - s[2]))       # -beta*DeltaH con H = -h S_z
        s = jnp.where(acepta, sp, s)
        return s, (s[2], acepta)
    _, (sz, acc) = lax.scan(paso, jnp.array([0.0, 0.0, 1.0]), random.split(key, N_CAL + N_PASOS))
    return sz[N_CAL:], acc[N_CAL:]

BH_PT = np.geomspace(10.0, 0.5, 8)            # ranura 0 = la más fría, como en el simulador
COS_PT = jnp.full(8, math.cos(math.pi / 2))

def cadena_pt(key):
    def paso(carry, k):
        s, cnt = carry
        k1, k2, k3 = random.split(k, 3)
        sp = sim.propuesta_cono(k1, s, COS_PT)
        acepta = random.uniform(k2, (8,)) < jnp.exp(jnp.asarray(BH_PT, jnp.float32) * (sp[:, 2] - s[:, 2]))
        s = jnp.where(acepta[:, None], sp, s)
        perm, _ = sim.permutacion_intercambio(-s[:, 2][None, :, None], jnp.asarray(BH_PT, jnp.float32), k3, cnt % 2)
        s = s[perm[0, :, 0]]
        return (s, cnt + 1), s[:, 2]
    s0 = jnp.tile(jnp.array([0.0, 0.0, 1.0]), (8, 1))
    _, sz = lax.scan(paso, (s0, jnp.int32(0)), random.split(key, N_CAL + N_PASOS))
    return sz[N_CAL:]

def resumen(sz, exacto):
    taus = tau_int_sokal_np(sz.T)
    se = math.sqrt(np.mean(taus * sz.var(axis=1) / sz.shape[1]) / sz.shape[0])
    est = sz.mean()
    return est, se, float(np.median(taus)), math.erfc(abs(est - exacto) / se / math.sqrt(2))

langevin = lambda x: 1 / math.tanh(x) - 1 / x
filas = []
for metodo, cos_a in (("uniforme", -1.0), ("cono π/4", math.cos(math.pi / 4))):
    f = jax.jit(jax.vmap(cadena, in_axes=(0, None, None)))
    for j, bh in enumerate(BH):
        sz, acc = f(random.split(random.PRNGKey(1000 + j), N_CAD), bh, cos_a)
        sz = np.asarray(sz)
        est, se, tau, p = resumen(sz, langevin(bh))
        filas.append([metodo, bh, langevin(bh), est, se, tau, float(np.asarray(acc).mean()), p])
sz_pt = np.asarray(jax.jit(jax.vmap(cadena_pt))(random.split(random.PRNGKey(2000), N_CAD)))   # (cadenas, pasos, 8)
for t, bh in enumerate(BH_PT):
    est, se, tau, p = resumen(sz_pt[:, :, t], langevin(bh))
    filas.append(["intercambio, ranura %d" % t, round(float(bh), 3), langevin(bh), est, se, tau, float("nan"), p])

pv = np.array([f[-1] for f in filas]); orden = np.argsort(pv); m = len(pv); p_holm = np.empty(m); acum = 0.0
for r_, i in enumerate(orden):
    acum = max(acum, min(1.0, (m - r_) * pv[i])); p_holm[i] = acum
tabla_E2 = "| Método | βh | Exacto | Estimado | Error estándar | τ_int (mediana) | Aceptación | p ajustado (Holm) |\n|---|---|---|---|---|---|---|---|\n"
tabla_E2 += "\n".join(f"| {f[0]} | {f[1]} | {f[2]:.5f} | {f[3]:.5f} | {f[4]:.5f} | {f[5]:.2f} | {f[6]:.3f} | {ph:.3f} |" for f, ph in zip(filas, p_holm))
open("tablas/E2.md", "w").write(tabla_E2); display(Markdown(tabla_E2))
print("Verificación E2:", "se cumple" if np.all(p_holm > ALPHA) else "no se cumple")

### Guía de lectura: algoritmo frente a la solución de Langevin

**Qué se hizo.** Se ejecutó el algoritmo de Metropolis sobre un único espín en un campo, $H=-hS_z$, cuya magnetización media exacta es la función de Langevin, $\langle S_z\rangle=\coth(\beta h)-1/(\beta h)$. Se probaron tres variantes con 64 cadenas independientes cada una: propuesta uniforme en la esfera, propuesta en cono ($\alpha=\pi/4$) y una escalera de 8 temperaturas con intercambio de réplicas que usa la función del simulador. Para cada una de las 20 comparaciones se tabularon la estimación, el valor exacto, el error estándar corregido por autocorrelación, el estadístico $z$ y los $p$-valores crudo y ajustado por Holm.

**Por qué es la prueba adecuada.** La solución es exacta, de modo que cualquier discrepancia significativa proviene del algoritmo y no de otra aproximación. El sistema no tiene vecinos, así que la prueba aísla la regla de aceptación, la simetría de la propuesta y el intercambio de réplicas de otras partes del código. Es sensible a errores característicos: un jacobiano omitido, un signo en $\Delta H$ o una propuesta asimétrica desplazan $\langle S_z\rangle$ de forma sistemática. Se usan varios valores de $\beta h$ para cubrir desde el régimen casi desordenado hasta el casi alineado, donde las dificultades de la propuesta son distintas. El error estándar incluye $\tau_{\mathrm{int}}$ porque las muestras sucesivas de cada cadena están correlacionadas, y la corrección de Holm controla las falsas alarmas al hacer 20 comparaciones simultáneas.

**Qué representa el resultado y qué se espera.** La tabla indica, para cada variante y cada valor de $\beta h$, si el promedio simulado es estadísticamente compatible con el exacto. Si el algoritmo es correcto, los $z$ se distribuyen aproximadamente como una normal estándar, sin patrón con $\beta h$ ni con la variante, y ningún $p$ ajustado cae por debajo de 0.05. Un patrón sistemático en el signo de $z$ señala un sesgo aunque ningún valor aislado sea significativo. La prueba no sustituye la verificación de la energía de la red ni del tablero de ajedrez: valida el núcleo del algoritmo, no su aplicación al disco.


In [ ]:
filas = []
for et, d in PRINCIPAL.items():
    filas.append(f"| {et} | {int(d['n_spins'])} | {int(d['nt'])} | {int(d['copias'])} | {int(d['barridos_term'])} | "
                 f"{'se cumple' if bool(d['puerta_ok']) else 'NO SE CUMPLE'} |")
tabla_puerta = ("| Estado | Espines | Temperaturas | Copias | Barridos de termalización | Puerta de equilibrio |\n"
                "|---|---|---|---|---|---|\n" + "\n".join(filas))
open("tablas/E0_puerta.md", "w").write(tabla_puerta); display(Markdown(tabla_puerta))

# Una fila por estado; en cada fila, tres temperaturas: una fria, una intermedia y una caliente de la escalera
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for fila, (et, d) in zip(ax, PRINCIPAL.items()):
    th = d["term_hist"].astype(np.float64); T = d["T"]; fria = d["arranque_frio"]
    for a, tt in zip(fila, (2.5, 6.0, 14.0)):
        t = int(np.argmin(np.abs(T - tt)))
        x = np.arange(1, th.shape[0] + 1) * int(d["barridos_term"]) / th.shape[0]   # barridos por trozo de termalización
        for r_ in range(th.shape[2]):
            a.plot(x, th[:, t, r_] / float(d["n_spins"]), c="tab:blue" if fria[r_] else "tab:red", alpha=0.6, lw=0.8)
        a.set_title(f"Estado {et}, T = {T[t]:.2f} K (azul: arranque ordenado, rojo: desordenado)", fontsize=8)
        a.set_xlabel("barridos de termalización"); a.set_ylabel("H / N (meV)")
plt.tight_layout(); plt.savefig("figures/E0_termalizacion.png", dpi=150); plt.show()


## Autocorrelación, tamaño de bloque, aceptación y convergencia de los datos de la simulación

Esta sección aplica a las series reales de la simulación las herramientas de la sección de incertidumbre. Cada serie corresponde a un estado de parámetros, una temperatura y una copia, y tiene $n=8000$ muestras guardadas, una cada 2 barridos. Por tanto, todo tiempo de autocorrelación se expresa en unidades de 2 barridos.

**Tiempo integrado de autocorrelación.** Para cada (estado, temperatura, copia) se calcula $\tau_{\mathrm{int}}$ de cuatro series: la energía $H$, la magnetización $m$ y las desviaciones cuadráticas $(H-\bar H)^2$ y $(m-\bar m)^2$. Las dos últimas intervienen porque $c$ y $\chi$ son varianzas (véase la sección de observables): su error depende de qué tan rápido se decorrelacionan los cuadrados de las desviaciones, y esto puede ser más lento que para $H$ y $m$. Se conserva el máximo de los cuatro. Se espera que $\tau_{\mathrm{int}}$ crezca cerca de la transición, donde las fluctuaciones son lentas, y que sea pequeño lejos de ella; cuando el intercambio de réplicas mezcla bien, ese crecimiento se atenúa pero no desaparece.

**Tamaño de bloque.** Con ese $\tau$ se elige $b^*$ según la regla $b\ge10\tau$ y $g=n/b\ge20$. Una temperatura sin $b^*$ disponible (porque $\tau>n/200=10$ muestras) se marca **no confiable** y no recibe error jackknife. Lo que se tabula es la fracción de temperaturas confiables y el $b^*$ elegido; una caída de esa fracción alrededor del pico de $c$ indica que ahí la corrida es demasiado corta para el error jackknife.

**Comprobación por meseta.** El error jackknife a $b^*$ se compara con $\sqrt{\tau\hat\sigma^2/n}$. Para un estimador tipo media, ambos deben coincidir salvo ruido del orden del 16 % esperado con $g=20$. Una discrepancia sistemática indica que $\tau$ está subestimado o que el bloque aún es corto.

**Error interno frente a dispersión entre copias.** El error interno de $c$ (jackknife dentro de cada copia) se compara con la desviación estándar de $c$ entre copias. Si las copias muestrean la misma distribución, ambas cantidades miden la variabilidad de un estimador calculado con una sola copia y deben ser parecidas. Una dispersión entre copias mayor que el error interno señala copias que no muestrean lo mismo (falta de equilibrio, estados metaestables) o un error interno subestimado; lo contrario sugiere un error interno sobreestimado.

**Aceptación local del Metropolis.** El ángulo del cono se ajusta durante la termalización hacia una aceptación de 0.5 y se fija en la medición. La aceptación medida debe ser cercana a 0.5, salvo donde el ángulo alcanza uno de sus límites ($0.01$ o $\pi$): a temperatura alta el ángulo tope $\pi$ no puede aumentar más, y a muy baja temperatura el mínimo puede impedir llegar al objetivo. Las desviaciones en esos extremos son esperables; en el resto del rango indican que la adaptación no convergió.


In [ ]:
def jk_media(x, b):
    # error jackknife por bloques de la media; x: (n, K)
    n = (x.shape[0] // b) * b; g = n // b
    s = x[:n].reshape(g, b, -1).sum(1); S = s.sum(0)
    mj = (S[None] - s) / (n - b)
    return np.sqrt((g - 1) / g * ((mj - mj.mean(0)) ** 2).sum(0))

def jk_var(x, b):
    # error jackknife por bloques de la varianza (divisor n), centrado para estabilidad; x: (n, K)
    n = (x.shape[0] // b) * b; g = n // b
    xc = x[:n] - x[:n].mean(0)
    s1 = xc.reshape(g, b, -1).sum(1); s2 = (xc ** 2).reshape(g, b, -1).sum(1)
    m = (s1.sum(0)[None] - s1) / (n - b); v = (s2.sum(0)[None] - s2) / (n - b) - m ** 2
    return np.sqrt((g - 1) / g * ((v - v.mean(0)) ** 2).sum(0))

E3 = {}
filas = []
for et, d in PRINCIPAL.items():
    H = d["H"].astype(np.float64); m = d["m"].astype(np.float64)
    n, NT_, R_ = H.shape; T = d["T"]
    LARGOS = np.array([b for b in (1, 2, 4, 5, 8, 10, 20, 25, 40, 50, 100, 125, 200, 250, 500) if n % b == 0])
    Hf, mf = H.reshape(n, -1), m.reshape(n, -1)
    taus = [tau_int_sokal_np(y) for y in (Hf, mf, (Hf - Hf.mean(0)) ** 2, (mf - mf.mean(0)) ** 2)]
    tau_max = np.maximum.reduce(taus).reshape(NT_, R_)
    tau_t = np.median(tau_max, axis=1)
    ok = LARGOS[None, :] >= 10 * tau_t[:, None]
    ib = np.where(ok.any(1), ok.argmax(1), len(LARGOS) - 1)
    b_star = LARGOS[ib]
    confiable = ok.any(1) & (n // b_star >= 20)
    se_b = np.stack([np.median(jk_media(Hf, b).reshape(NT_, R_), axis=1) for b in LARGOS], 1)   # (NT, nb)
    se_ref = np.median(np.sqrt(taus[0] * Hf.var(0) / n).reshape(NT_, R_), axis=1)
    coc = se_b[np.arange(NT_), ib] / se_ref
    # error interno del calor específico de cada copia (jackknife con b*) contra la dispersión entre copias
    f_c = 1.0 / (float(d["n_spins"]) * float(d["kB"]) ** 2 * T ** 2)
    c_rep = H.var(0) * f_c[:, None]                                        # (NT, R)
    se_int = np.array([np.sqrt(np.mean(jk_var(H[:, t], b_star[t]) ** 2)) * f_c[t] for t in range(NT_)])
    se_ext = c_rep.std(1, ddof=1)
    acc = d["aceptacion"].mean(1); alfa = d["alfa"]
    E3[et] = dict(T=T, tau=tau_t, b=b_star, conf=confiable, coc=coc, se_b=se_b, se_ref=se_ref, largos=LARGOS,
                  acc=acc, alfa=alfa, se_int=se_int, se_ext=se_ext, c_rep=c_rep)
    for t in list(range(0, NT_, max(1, NT_ // 8))) + [int(np.argmax(c_rep.mean(1)))]:
        filas.append(f"| {et} | {T[t]:.2f} | {acc[t]:.2f} | {alfa[t]:.3f} | {tau_t[t]:.1f} | {b_star[t]} | {n // b_star[t]} | "
                     f"{'sí' if confiable[t] else 'no'} | {coc[t]:.2f} | {se_int[t] / se_ext[t]:.2f} |")
tabla_E3 = ("| Estado | T (K) | Aceptación | α (rad) | τ_int (×10 barridos) | b* | g | Confiable | se_JK(b*)/√(τσ²/n) | se interno / se entre copias de c |\n"
            "|---|---|---|---|---|---|---|---|---|---|\n" + "\n".join(filas))
open("tablas/E3.md", "w").write(tabla_E3); display(Markdown(tabla_E3))

# Criterio de E3: la fracción de temperaturas con jackknife confiable se evalúa en la ventana de T*; el total se informa aparte.
filas_resumen = []
for et, r in E3.items():
    ven = (r["T"] >= VENTANA_T[0]) & (r["T"] <= VENTANA_T[1])
    f_ven, f_tot = float(r["conf"][ven].mean()), float(r["conf"].mean())
    filas_resumen.append(f"| {et} | {f_ven:.3f} | {f_tot:.3f} | {'se cumple' if f_ven >= 0.9 else 'no se cumple'} |")
tabla_E3r = ("| Estado | Fracción confiable en la ventana 6–20 K | Fracción confiable en toda la escalera | Criterio (≥ 90 % en la ventana) |\n"
             "|---|---|---|---|\n" + "\n".join(filas_resumen))
open("tablas/E3_resumen.md", "w").write(tabla_E3r); display(Markdown(tabla_E3r))

for et, r in E3.items():
    libre = r["alfa"] < np.pi * 0.999
    ok_acc = np.all((r["acc"][libre] > 0.3) & (r["acc"][libre] < 0.7))
    ven = (r["T"] >= VENTANA_T[0]) & (r["T"] <= VENTANA_T[1])
    frac_ven = r["conf"][ven].mean(); frac_tot = r["conf"].mean()
    c = r["coc"][r["conf"]]; frac_coc = np.mean((c > 0.8) & (c < 1.2)) if c.size else float("nan")
    print(f"Estado {et}: aceptación en [0.3, 0.7] donde α<π: {ok_acc} | confiables en la ventana {frac_ven:.0%} "
          f"y en toda la escalera {frac_tot:.0%} | cociente de meseta en [0.8, 1.2]: {frac_coc:.2f}")
    print(f"Verificación E3 (estado {et}):", "se cumple" if ok_acc and frac_ven >= 0.9 and frac_coc >= 0.9 else "no se cumple")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for et, r in E3.items():
    ax[0].plot(r["T"], r["acc"], label=f"estado {et}")
    ax[1].plot(r["T"], r["alfa"], label=f"estado {et}")
    ax[2].plot(r["T"], r["tau"], label=f"estado {et}")
ax[0].axhline(0.5, ls="--", c="gray"); ax[0].set_ylabel("aceptación local"); ax[0].set_ylim(0, 1)
ax[1].set_ylabel("ángulo del cono α (rad)"); ax[1].set_yscale("log")
ax[2].set_ylabel("τ_int (×10 barridos)"); ax[2].set_yscale("log")
for a in ax: a.set_xlabel("T (K)"); a.set_xscale("log"); a.legend()
plt.tight_layout(); plt.savefig("figures/E3_aceptacion_tau.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (et, r) in zip(ax, E3.items()):
    for t in sorted({0, int(np.argmax(r["c_rep"].mean(1))), len(r["T"]) - 1}):
        l, = a.plot(r["largos"], r["se_b"][t], "o-", label=f"T = {r['T'][t]:.2f} K")
        a.axhline(r["se_ref"][t], ls="--", c=l.get_color(), alpha=0.6)
    a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel("largo de bloque b"); a.set_ylabel("se jackknife de H̄ (meV)")
    a.set_title(f"Estado {et}: meseta (punteada: √(τσ²/n))"); a.legend()
plt.tight_layout(); plt.savefig("figures/E3_meseta.png", dpi=150); plt.show()

### Guía de lectura: autocorrelación, aceptación, jackknife y meseta

Tablas y figuras: `E3.md`, `E3_resumen.md`, `E3_aceptacion_tau.png` y `E3_meseta.png`.

**Qué se hizo y qué representa.** Para cada estado, temperatura y copia se calculó el tiempo integrado de autocorrelación $\tau_{\mathrm{int}}$ de $H$, $m$, $(H-\bar H)^2$ y $(m-\bar m)^2$ mediante FFT con ventana automática de Sokal, y se conservó el máximo. A partir de él se eligió el tamaño de bloque $b^*$ del jackknife (menor $b$ con $b\ge10\tau$ y al menos 20 bloques, lo que con $n=8\,000$ exige $\tau_{\mathrm{int}}\le40$ muestras). Se registró la aceptación del Metropolis, se comparó el error jackknife en $b^*$ con $\sqrt{\tau\hat\sigma^2/n}$ y se contrastó el error interno de $c$ con su dispersión entre copias. `E3.md` tabula estas cantidades a lo largo de la escalera, y `E3_resumen.md` da, por estado, la fracción de temperaturas con jackknife confiable en la ventana 6–20 K y en toda la escalera. `E3_aceptacion_tau.png` muestra la aceptación local y $\tau_{\mathrm{int}}$ frente a $T$, y `E3_meseta.png` el error jackknife en función de $b$.

**Por qué es la prueba adecuada.** Las muestras de una cadena están correlacionadas, y la fórmula de muestras independientes subestima el error en un factor $\sqrt{\tau}$. El jackknife por bloques corrige esa dependencia a condición de que los bloques sean largos frente a $\tau$ (para que el sesgo sea pequeño) y de que haya suficientes (para que el error estimado no sea ruidoso). La regla $b\ge10\tau$, $g\ge20$ equilibra ambos sesgos, y la comprobación por meseta es un control independiente: si $b^*$ está bien elegido, el error jackknife debe haberse estabilizado y coincidir con el valor teórico. Comparar el error interno con la dispersión entre copias pone a prueba el supuesto de que todas las copias muestrean la misma distribución. El máximo de los cuatro $\tau$ se usa porque $c$ y $\chi$ son varianzas, cuyo error depende de la serie más lenta.

**Cómo leer el resultado.**

- *$\tau_{\mathrm{int}}$:* mide cuántas muestras sucesivas equivalen a una independiente. Se espera mayor cerca de la transición y menor lejos de ella.
- *Aceptación local:* debe aproximarse al objetivo de 0.5 salvo donde el ángulo del cono llega a un límite.
- *Marca «no confiable»:* indica que no existe $b^*$ (la serie es demasiado corta para un error jackknife fiable). `E3_resumen.md` cuantifica la fracción de temperaturas confiables; la lectura principal es la de la ventana, y el total se muestra aparte.
- *Meseta:* un error jackknife que coincide con el valor teórico respalda las cifras; una discrepancia sistemática indica $\tau$ subestimado.
- *Error interno frente a dispersión entre copias:* un error interno mucho menor que la dispersión sugiere que las copias no están en equilibrio o que $b^*$ es insuficiente, y obliga a leer con cautela las bandas de la sección siguiente.


## Intercambio de réplicas y convergencia

Esta sección tabula los diagnósticos del intercambio de réplicas y de la convergencia de las cadenas, definidos en la sección de criterios de convergencia. Para cada estado de parámetros se resume la escalera completa de temperaturas entre 0.5 y 20 K y, de forma separada, la ventana $6\le T\le20$ K en la que se evalúa el equilibrio.

**Diagnósticos y umbrales.**

- **Aceptación del intercambio** $\ge0.1$ en todos los pares de ranuras vecinas. Por debajo de ese valor, un par actúa como cuello de botella: las configuraciones calientes no llegan a las ranuras frías y estas conservan la memoria del arranque. Se tabula el mínimo sobre los pares y la ranura donde ocurre, que suele estar cerca del pico de $c$.
- **Viajes redondos totales** (termalización más medición) $\ge10$ por copia (mediana entre copias) y $\ge0.5$ veces la predicción de la teoría del intercambio no reversible, $(\text{intentos})/\big[2\big(1+\sum_tr_t/(1-r_t)\big)\big]$ con $r_t=1-a_t$ la tasa de rechazo del par $t$ y los intentos de ambas fases. Un viaje redondo es una configuración que desciende a la ranura fría, sube a la caliente y vuelve. Observar mucho menos que la teoría indica que la mezcla es peor de lo que sugieren las aceptaciones; observar mucho más sugiere un error de conteo, razón por la cual la autoprueba con aceptación forzada precede a la corrida.
- **$\hat R$ dividido** de Gelman–Rubin $<1.01$ en la energía de cada temperatura. Compara la variación entre las mitades de las copias con la variación dentro de ellas, y detecta tanto diferencias entre copias como deriva dentro de una copia.
- **Prueba de Welch–Holm** con $p>0.05$ (valor ajustado): sin diferencias en la energía media entre las copias que arrancaron ordenadas y las que arrancaron desordenadas. La corrección de Holm controla las falsas alarmas por haber una prueba en cada temperatura.
- **Puerta de equilibrio** de la termalización, evaluada solo para $T\ge6$ K: declara CUMPLE o NO CUMPLE, y cuántos barridos de termalización se emplearon (16 000 como base, hasta 32 000).

**Cómo se lee la tabla.** Las columnas de aceptación y viajes describen la escalera: dicen si la mezcla permite que una configuración recorra todas las temperaturas. Las de $\hat R$ y Welch–Holm describen cada temperatura: dicen si las copias, aun con arranques opuestos, coinciden. Los dos tipos se leen juntos. Una escalera con buena aceptación y pocos viajes revela un problema de mezcla global; una temperatura con $\hat R$ alto en una escalera sana apunta a una región con metaestabilidad. El veredicto de convergencia rige para la ventana 6–20 K; las cifras del total de la escalera se reportan aparte y documentan la zona fría no convergida.

**Marca por temperatura.** Cada temperatura de la ventana recibe la marca **convergió** únicamente si cumple todos los criterios aplicables a ella, y **provisional** si falla alguno. Las temperaturas provisionales y las de menos de 6 K se conservan en las figuras, señaladas, pero las conclusiones que dependen de ellas se enuncian con reserva.

**Qué se hizo y qué representa.** Se calcularon, para cada estado, los cinco diagnósticos anteriores y se resumieron en una tabla por estado. Representan una prueba de ausencia de evidencia contra el equilibrio: cada criterio detecta un modo de falla distinto (escalera cortada, mezcla lenta, copias discordantes, memoria del arranque, termalización incompleta). La combinación es adecuada porque ningún criterio individual prueba equilibrio.

**Qué se espera.** A temperaturas altas, donde la dinámica es rápida y las configuraciones se decorrelacionan, se espera que todos los criterios se cumplan. Las dificultades se concentran a temperaturas bajas, donde pueden existir estados metaestables, y alrededor del pico de $c$, donde la aceptación del intercambio es menor. En una corrida anterior con otro radio, el estado B mostró falta de equilibrio a baja temperatura; esto se cita solo como antecedente y no anticipa el resultado de esta corrida.


In [ ]:
def rhat_dividido(x):
    # x: (n, NT, R) -> R-hat de Gelman-Rubin con cada copia partida en dos mitades (2R cadenas)
    h = x.shape[0] // 2
    c = np.concatenate([x[:h], x[h:2 * h]], axis=2)
    W = c.var(0, ddof=1).mean(-1); Bm = h * c.mean(0).var(-1, ddof=1)
    return np.sqrt(((h - 1) / h * W + Bm / h) / W)

def holm(p):
    p = np.asarray(p); orden = np.argsort(p); m = len(p); out = np.empty(m); acum = 0.0
    for r_, i in enumerate(orden):
        acum = max(acum, min(1.0, (m - r_) * p[i])); out[i] = acum
    return out

PT = {}
filas = []
for et, d in PRINCIPAL.items():
    H = d["H"].astype(np.float64); T = d["T"]; R_ = H.shape[2]; NT_ = len(T)
    sw = d["sw_acc"] / np.maximum(d["sw_int"] * R_, 1)
    viajes = d["viajes"].astype(float) + d["viajes_term"].astype(float)   # recorrido completo por copia
    swc = np.clip(sw, 1e-3, 0.999)
    intentos = float(d["intentos_term"]) + float(d["intentos_med"])
    esperados = intentos / (2 * (1 + np.sum((1 - swc) / swc)))   # teoría del intercambio no reversible
    en_ventana = (T >= VENTANA_T[0]) & (T <= VENTANA_T[1])
    rh = rhat_dividido(H)
    fria = d["arranque_frio"]
    medias = H.mean(0)                                                     # (NT, R)
    p_fc = np.array([stats.ttest_ind(medias[t, fria], medias[t, ~fria], equal_var=False).pvalue for t in range(NT_)])
    p_fc = holm(np.nan_to_num(p_fc, nan=1.0))
    PT[et] = dict(T=T, sw=sw, viajes=viajes, rhat=rh, p_fc=p_fc, en_ventana=en_ventana,
              conv=(rh < 1.01) & (p_fc > ALPHA), esperados=esperados)
    c1 = sw.min() >= 0.1
    c2 = bool(np.median(viajes) >= 0.5 * esperados and np.median(viajes) >= 10)
    conv = (rh < 1.01) & (p_fc > ALPHA)
    c3 = bool(np.all(conv[en_ventana]))
    filas.append(f"| {et} | {sw.min():.2f} (en T={T[int(np.argmin(sw))]:.2f} K) | {np.median(viajes):.0f} (teoría {esperados:.0f}) | "
                 f"{rh[en_ventana].max():.3f} / {rh.max():.3f} | {int((rh[en_ventana] >= 1.01).sum())} / {int((rh >= 1.01).sum())} | "
                 f"{int((p_fc[en_ventana] <= ALPHA).sum())} / {int((p_fc <= ALPHA).sum())} | "
                 f"{'se cumple' if c1 and c2 and c3 else 'no se cumple'} |")
    print(f"Estado {et}: aceptación mínima {sw.min():.2f} | viajes por copia (mediana) {np.median(viajes):.0f} "
          f"frente a la teoría {esperados:.0f} | R̂ máximo {rh[en_ventana].max():.3f} en la ventana y {rh.max():.3f} en total | "
          f"convergencia en la ventana: {'se cumple' if c3 else 'no se cumple'}")
tabla_PT = ("| Estado | Aceptación mínima de intercambio | Viajes totales por copia (mediana; teoría) | R̂ máximo (ventana / total) | "
            "T con R̂ ≥ 1.01 (ventana / total) | T con fríos ≠ calientes, Holm (ventana / total) | Convergencia en la ventana |\n"
            "|---|---|---|---|---|---|---|\n" + "\n".join(filas))
open("tablas/E3_intercambio.md", "w").write(tabla_PT); display(Markdown(tabla_PT))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for et, r in PT.items():
    Tm = np.sqrt(r["T"][:-1] * r["T"][1:])
    ax[0].plot(Tm, r["sw"], label=f"estado {et}"); ax[1].plot(r["T"], r["rhat"], label=f"estado {et}")
ax[0].axhline(0.1, ls="--", c="gray"); ax[0].set_ylabel("aceptación del intercambio (t, t+1)")
ax[1].axhline(1.01, ls="--", c="gray"); ax[1].set_ylabel("R̂ dividido de la energía")
for a in ax:
    a.axvspan(*VENTANA_T, color="gray", alpha=0.15, label="ventana 6–20 K")
    a.set_xscale("log"); a.set_xlabel("T (K)"); a.legend()
plt.tight_layout(); plt.savefig("figures/E3_intercambio_rhat.png", dpi=150); plt.show()

### Guía de lectura: intercambio de réplicas y convergencia

Tablas y figuras: `E0_puerta.md`, `E3_intercambio.md`, `E0_termalizacion.png` y `E3_intercambio_rhat.png`.

**Qué se hizo y qué representa.** Para cada estado se tabularon los diagnósticos de la escalera y de las cadenas. `E0_puerta.md` resume el estado, el número de espines, de temperaturas y de copias, los barridos de termalización empleados y si la puerta de equilibrio se cumple o NO SE CUMPLE (evaluada solo en 6–20 K). `E3_intercambio.md` contiene, por estado: la aceptación mínima de intercambio; los viajes redondos totales por copia (mediana) frente a la teoría; el $\hat R$ máximo en la ventana y en total; el número de temperaturas con $\hat R\ge1.01$ (ventana y total); el número de temperaturas con arranques distintos por Welch–Holm (ventana y total); y el veredicto de convergencia en la ventana. La figura `E0_termalizacion.png` muestra la evolución de la energía media por trozo durante la termalización, y `E3_intercambio_rhat.png` muestra $\hat R$ en función de $T$ con las temperaturas no convergidas sombreadas.

**Por qué es la prueba adecuada.** Un error estándar pequeño no garantiza que las cadenas hayan alcanzado el equilibrio: varias cadenas atrapadas en el mismo estado metaestable darían medias muy estables y erróneas. Por eso las copias arrancan en condiciones opuestas y se comprueba que lo olviden. $\hat R$ detecta diferencias entre cadenas y deriva dentro de una cadena; Welch compara de forma directa los dos tipos de arranque, y Holm controla las falsas alarmas de una prueba por temperatura sin suponer independencia entre ellas. La aceptación y los viajes redondos totales evalúan algo distinto: si la escalera deja que las configuraciones recorran todas las temperaturas. Cada criterio detecta un tipo de falla que los demás pueden pasar por alto, razón por la cual se combinan.

**Cómo leer cada columna.**

- *Aceptación mínima:* debe ser $\ge0.1$. Un valor menor señala un cuello de botella de la escalera, normalmente cerca del pico de $c$.
- *Viajes totales frente a teoría:* se cumple si la mediana es $\ge10$ y $\ge0.5$ veces la predicción. Una cifra muy baja indica mezcla lenta aunque la aceptación sea aceptable.
- *$\hat R$ máximo y número de temperaturas con $\hat R\ge1.01$:* $\hat R\ge1.01$ significa que la variación entre medias de cadena no es explicable por el ruido intra-cadena; las copias no son réplicas de la misma distribución. Se lee el valor dentro de la ventana, que decide el veredicto, y el total, que documenta la zona fría no convergida.
- *Welch–Holm:* un número positivo de temperaturas con arranques distintos indica memoria del arranque (histéresis o atasco). Con 8 copias por grupo, $p>0.05$ no demuestra igualdad.

**Qué se espera.** Las temperaturas altas, donde la dinámica es rápida, deberían cumplir todo; las dificultades se concentran a baja temperatura y alrededor del pico del calor específico. La ventana 6–20 K puede cumplir el veredicto aunque el total no lo haga, porque la zona fría queda fuera del criterio. La marca limita el alcance de las conclusiones: los intervalos de las temperaturas provisionales describen la variabilidad entre copias y no certifican el equilibrio, y la ausencia de alarmas es evidencia parcial pero no una prueba de convergencia.


## Bandas de $c(T)$ y $\chi(T)$ y distribución de $T^*$

Las 16 copias son independientes entre sí, de modo que el bootstrap clásico es válido a ese nivel y no dentro de cada cadena (véase la sección de incertidumbre). Las bandas de $c(T)$ y $\chi(T)$ y la distribución de $T^*$ se construyen con $B=2000$ remuestras de copias con reemplazo.

**Remuestreo.** Los índices de las copias se generan por inversión discreta, $I=\lfloor RU\rfloor$ con $U\sim\mathcal U[0,1)$ y $R$ el número de copias. En cada remuestra se agrupan las series de las copias elegidas y se recalcula la varianza. La forma eficiente no concatena series sino que suma, por copia, las cantidades $(\sum1,\sum H,\sum H^2)$ (y las análogas para $m$), multiplicadas por el número de veces que la copia fue elegida. Con ellas se obtienen media y varianza del conjunto remuestreado sin recorrer las series de nuevo.

**Intervalos.** Para cada temperatura se calculan dos intervalos del 95 %:

- **Percentil:** cuantiles 2.5 % y 97.5 % de las 2000 réplicas. Es simple y transparente.
- **BCa:** corrige el sesgo mediano $z_0$ y la asimetría con el factor de aceleración $a$, calculado con las réplicas jackknife de la muestra original que dejan fuera una copia a la vez. Importa para varianzas, cuya distribución de muestreo es asimétrica.

Cuando los dos intervalos coinciden, la distribución bootstrap es aproximadamente simétrica y sin sesgo. Cuando difieren, BCa es la referencia y la discrepancia mide cuánto sesgo y asimetría había. Las bandas son **puntuales** (95 % en cada temperatura por separado) y no simultáneas: no son una banda de confianza para toda la curva.

**Repesado y rejilla fina.** Las curvas se evalúan en una rejilla de 0.01 K por repesado de histogramas (Ferrenberg–Swendsen, 1988), cada punto desde la ranura más cercana en $\beta$. En cada punto se verifica que el tamaño efectivo de Kish de los pesos alcance al menos el 10 % de las muestras.

**Estimador agrupado e intra-copia.** El estimador agrupado de $c$ usa la varianza de todas las muestras mezcladas de las 16 copias y, por tanto, incluye la varianza entre copias; el intra-copia promedia las varianzas calculadas dentro de cada copia. Cuando las copias están en equilibrio coinciden; cuando están atrapadas en mínimos distintos, el agrupado sobrestima $c$. La curva intra-copia se muestra punteada en la figura para señalar esa diferencia.

**Temperatura del pico.** $T^*$ se define **solo con el calor específico**: es el argmax de $c(T)$ en la ventana 6–20 K, que cubre la región de la transición orden–desorden. En cada remuestra se recalcula la curva repesada completa y se registra su argmax; las 2000 réplicas forman la distribución de $T^*$, que se resume con la mediana, el error estándar bootstrap y el intervalo percentil del 95 %. Como el pico es plano, se reporta también la **meseta** $[T_{lo},T_{hi}]$, el intervalo de temperaturas cuyo $c$ es compatible con el del pico. La susceptibilidad $\chi(T)$ no define $T^*$: se muestra su curva y se indica si su máximo en la ventana cae en un borde.

**Qué se hizo y qué representa.** Se remuestrearon las copias para trasladar la incertidumbre de la simulación a $c(T)$, a $\chi(T)$ y a $T^*$. Las bandas representan la variabilidad entre copias independientes; la distribución de $T^*$ representa qué tan bien determinada queda la posición de un máximo plano. El procedimiento es correcto porque la copia es la unidad independiente, y porque el argmax de una curva aplanada se estudia mediante su distribución en lugar de propagar errores puntuales.

**Cómo se lee.** Una banda estrecha indica que las copias coinciden en $c$ a esa temperatura; una banda ancha, que no. La distribución de $T^*$ debe leerse con la forma de $c$: un pico plano da una distribución ancha, y dos máximos de altura comparable, una distribución bimodal. Un $T^*$ pegado a 6 o 20 K indica que el pico real está fuera de la ventana. Todos estos intervalos miden variabilidad entre copias; aplican a lo que la marca de convergencia declara como confiable, y para las temperaturas provisionales o menores de 6 K se interpretan con reserva.


In [ ]:
NBOOT = 2000

def Phi(x):
    return 0.5 * (1.0 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2.0)))

def Phi_inv(p):
    # inversa de la normal estándar por bisección sobre Phi (solo usa erf)
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1 - 1e-12)
    lo, hi = np.full_like(p, -10.0), np.full_like(p, 10.0)
    for _ in range(80):
        mid = 0.5 * (lo + hi); menor = Phi(mid) < p
        lo = np.where(menor, mid, lo); hi = np.where(menor, hi, mid)
    return 0.5 * (lo + hi)

def multiplicidades(R, B=NBOOT, semilla=2024):
    # índices por inversión discreta I = floor(R U) y conteo de cuántas veces sale cada copia
    U = np.random.default_rng(semilla).random((B, R))
    I = np.minimum(np.floor(R * U).astype(int), R - 1)
    W = np.zeros((B, R)); np.add.at(W, (np.repeat(np.arange(B), R), I.ravel()), 1.0)
    return W

def var_agrupada(W, S):
    # W: (B, R) multiplicidades; S: (nT, 3, R) sumas (peso, peso*y, peso*y^2) por copia -> varianza agrupada (B, nT)
    P = np.einsum("br,tkr->btk", W, S)
    return P[..., 2] / P[..., 0] - (P[..., 1] / P[..., 0]) ** 2

def sumas(x):
    # x: (n, nT, R) -> (nT, 3, R), centrado por temperatura para estabilidad numérica
    y = x - x.mean(axis=(0, 2), keepdims=True)
    return np.stack([np.full(y.shape[1:], y.shape[0], float), y.sum(0), (y * y).sum(0)], axis=1)

def intervalos(est, boots, jk, alpha=0.05):
    lo_p, hi_p = np.quantile(boots, [alpha / 2, 1 - alpha / 2], axis=0)
    jbar = jk.mean(0)
    num = np.sum((jbar - jk) ** 3, 0); den = 6.0 * np.sum((jbar - jk) ** 2, 0) ** 1.5
    a = np.where(den > 0, num / np.where(den > 0, den, 1.0), 0.0)
    z0 = Phi_inv(np.mean(boots < est[None], 0))
    q = [Phi(z0 + (z0 + z) / (1 - a * (z0 + z))) for z in Phi_inv(np.array([alpha / 2, 1 - alpha / 2]))]
    lo_b = np.array([np.quantile(boots[:, t], q[0][t]) for t in range(boots.shape[1])])
    hi_b = np.array([np.quantile(boots[:, t], q[1][t]) for t in range(boots.shape[1])])
    return lo_p, hi_p, lo_b, hi_b

def curvas_repesadas(d, paso=0.01):
    # c(T) y chi(T) en una rejilla fina, repesando desde la ranura más cercana en beta.
    # Devuelve la rejilla, las sumas por copia (nF, 3, R) para H y m, y el tamaño efectivo relativo.
    H = d["H"].astype(np.float64); m = d["m"].astype(np.float64); T = d["T"]; kB_ = float(d["kB"])
    Tf = np.round(np.arange(T[0], T[-1], paso), 4); bf = 1.0 / (kB_ * Tf); bt = 1.0 / (kB_ * T)
    ranura = np.abs(bf[:, None] - bt[None]).argmin(1)
    SH = np.zeros((len(Tf), 3, H.shape[2])); Sm = np.zeros_like(SH); ess = np.zeros(len(Tf))
    for t in np.unique(ranura):
        k = np.nonzero(ranura == t)[0]
        y = H[:, t] - H[:, t].mean(); mt = m[:, t]
        lw = -(bf[k] - bt[t])[:, None, None] * y[None]
        lw -= lw.max(axis=(1, 2), keepdims=True); w = np.exp(lw)
        SH[k] = np.stack([w.sum(1), (w * y).sum(1), (w * y * y).sum(1)], 1)
        Sm[k] = np.stack([w.sum(1), (w * mt).sum(1), (w * mt * mt).sum(1)], 1)
        ess[k] = w.sum((1, 2)) ** 2 / (w ** 2).sum((1, 2)) / y.size
    return Tf, SH, Sm, ess

def analizar_Tstar(d, B=NBOOT):
    N, kB_ = float(d["n_spins"]), float(d["kB"])
    Tf, SH, Sm, ess = curvas_repesadas(d)
    fc, fx = 1.0 / (N * kB_ ** 2 * Tf ** 2), N / (kB_ * Tf)
    R = SH.shape[2]; W = multiplicidades(R, B)
    out = dict(Tf=Tf, ess=ess)
    ven = (Tf >= VENTANA_T[0]) & (Tf <= VENTANA_T[1])
    for nombre, S, f in (("c", SH, fc), ("chi", Sm, fx)):
        est = var_agrupada(np.ones((1, R)), S)[0] * f
        boots = var_agrupada(W, S) * f
        out[nombre] = est
        out[nombre + "_ts"] = Tf[ven][boots[:, ven].argmax(1)]; out[nombre + "_t"] = Tf[ven][est[ven].argmax()]
        out[nombre + "_pico"] = est[ven].max(); out[nombre + "_glob"] = Tf[est.argmax()]
        out[nombre + "_se"] = boots.std(0, ddof=1)
        k_ven = np.flatnonzero(ven); k_p = int(k_ven[np.argmax(est[ven])])
        out[nombre + "_k_pico"] = k_p; out[nombre + "_pico_se"] = boots[:, k_p].std(ddof=1)
    return out

E4, filas_c, filas_chi, filas_intra = {}, [], [], []
for et, d in PRINCIPAL.items():
    T = d["T"]; N, kB_ = float(d["n_spins"]), float(d["kB"]); R = d["H"].shape[2]
    en_ventana = (T >= VENTANA_T[0]) & (T <= VENTANA_T[1]); idx_ven = np.flatnonzero(en_ventana)
    W = multiplicidades(R); Wjk = np.ones((R, R)) - np.eye(R)
    res = {}
    for nombre, x, f in (("c", d["H"].astype(np.float64), 1.0 / (N * kB_ ** 2 * T ** 2)), ("chi", d["m"].astype(np.float64), N / (kB_ * T))):
        S = sumas(x)
        est = var_agrupada(np.ones((1, R)), S)[0] * f; boots = var_agrupada(W, S) * f; jk = var_agrupada(Wjk, S) * f
        lo_p, hi_p, lo_b, hi_b = intervalos(est, boots, jk)
        se = boots.std(0, ddof=1)
        acuerdo = np.mean((np.abs(lo_p - lo_b) < se) & (np.abs(hi_p - hi_b) < se))
        res[nombre] = dict(est=est, se=se, lo_p=lo_p, hi_p=hi_p, lo_b=lo_b, hi_b=hi_b, acuerdo=acuerdo)
    rw = analizar_Tstar(d)
    # estimador intra-copia: la varianza de H se calcula dentro de cada copia y se promedia sobre las R copias
    c_intra = d["H"].astype(np.float64).var(0, ddof=1).mean(-1) / (N * kB_ ** 2 * T ** 2)
    k_pico = int(idx_ven[np.argmax(res["c"]["est"][en_ventana])])
    rc = res["c"]
    # meseta del pico: temperaturas contiguas, en la ventana, con c >= c_max - 1 se(c en el pico)
    umbral = rw["c_pico"] - rw["c_pico_se"]
    ven_f = (rw["Tf"] >= VENTANA_T[0]) & (rw["Tf"] <= VENTANA_T[1])
    lo_i = hi_i = int(rw["c_k_pico"])
    while lo_i > 0 and ven_f[lo_i - 1] and rw["c"][lo_i - 1] >= umbral:
        lo_i -= 1
    while hi_i + 1 < len(ven_f) and ven_f[hi_i + 1] and rw["c"][hi_i + 1] >= umbral:
        hi_i += 1
    meseta = (float(rw["Tf"][lo_i]), float(rw["Tf"][hi_i]))
    ts, ic = rw["c_ts"], np.quantile(rw["c_ts"], [0.025, 0.975])
    E4[et] = dict(T=T, res=res, rw=rw, c_intra=c_intra, c_pool=rc["est"], en_ventana=en_ventana, meseta=meseta)
    filas_c.append(f"| {et} | {T[k_pico]:.2f} K | {rw['c_t']:.2f} K | {np.median(ts):.2f} K | {float(ts.std(ddof=1)):.3f} K | "
                   f"[{ic[0]:.2f}, {ic[1]:.2f}] K | [{meseta[0]:.2f}, {meseta[1]:.2f}] K | "
                   f"{rc['est'][k_pico]:.4g} [{rc['lo_p'][k_pico]:.4g}, {rc['hi_p'][k_pico]:.4g}] / "
                   f"BCa [{rc['lo_b'][k_pico]:.4g}, {rc['hi_b'][k_pico]:.4g}] |")
    k_chi = int(idx_ven[np.argmax(res["chi"]["est"][en_ventana])])
    borde = bool(k_chi in (idx_ven[0], idx_ven[-1]))
    filas_chi.append(f"| {et} | {T[k_chi]:.2f} K | {'sí' if borde else 'no'} | "
                     f"{'sin pico interior' if borde else 'pico interior en la ventana'} |")
    k_star = int(np.argmin(np.abs(T - rw["c_t"]))); baja = (T >= 0.5) & (T <= 2.0)
    cp, ci = float(rc["est"][k_star]), float(c_intra[k_star])
    filas_intra.append(f"| {et} | {T[k_star]:.2f} K | {cp:.4g} | {ci:.4g} | {ci / cp if cp else float('nan'):.2f} | "
                       f"{rc['est'][en_ventana].mean():.4g} | {c_intra[en_ventana].mean():.4g} | "
                       f"{rc['est'][baja].mean():.4g} | {c_intra[baja].mean():.4g} |")
    ok_ess = rw["ess"].min() >= 0.1
    chi_msg = "sin pico interior" if borde else f"pico en {T[k_chi]:.2f} K"
    print(f"Estado {et}: tamaño efectivo mínimo de los pesos {rw['ess'].min():.2f} | acuerdo percentil/BCa de c: {rc['acuerdo']:.0%} | "
          f"T* = {rw['c_t']:.2f} K, IC95 [{ic[0]:.2f}, {ic[1]:.2f}] K, meseta [{meseta[0]:.2f}, {meseta[1]:.2f}] K | "
          f"χ en la ventana: {chi_msg}")
    print(f"Verificación E4 (estado {et}):", "se cumple" if ok_ess and rc["acuerdo"] >= 0.9 else "no se cumple")
tabla_E4 = ("| Estado | Pico de c en la escalera (ventana) | T* repesada | Mediana bootstrap de T* | se bootstrap de T* | IC 95 % de T* (percentil) | Meseta del pico [T_lo, T_hi] | c en el pico: IC percentil / BCa |\n"
            "|---|---|---|---|---|---|---|---|\n" + "\n".join(filas_c))
open("tablas/E4.md", "w").write(tabla_E4); display(Markdown(tabla_E4))
tabla_E4chi = ("| Estado | argmax de χ en la ventana | ¿Cae en un borde? | Veredicto |\n|---|---|---|---|\n" + "\n".join(filas_chi))
open("tablas/E4_chi.md", "w").write(tabla_E4chi); display(Markdown(tabla_E4chi))
tabla_E4intra = ("| Estado | T de referencia | c pooled en T* | c intra-copia en T* | Cociente en T* | Media pooled en la ventana 6–20 K | Media intra-copia en la ventana | Media pooled en 0.5–2 K | Media intra-copia en 0.5–2 K |\n"
                 "|---|---|---|---|---|---|---|---|---|\n" + "\n".join(filas_intra))
open("tablas/E4_intra.md", "w").write(tabla_E4intra); display(Markdown(tabla_E4intra))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for j, (et, e4) in enumerate(E4.items()):
    for i, nombre in enumerate(("c", "chi")):
        a = ax[i, j]; r = e4["res"][nombre]; rw = e4["rw"]
        a.axvspan(e4["T"].min(), VENTANA_T[0], color="0.85", alpha=0.6, label="zona no convergida (T < 6 K)")
        a.fill_between(e4["T"], r["lo_p"], r["hi_p"], alpha=0.35, label="IC 95 % percentil (escalera)")
        a.plot(e4["T"], r["lo_b"], "k:", lw=0.8, label="IC 95 % BCa"); a.plot(e4["T"], r["hi_b"], "k:", lw=0.8)
        a.plot(rw["Tf"], rw[nombre], lw=1, label="curva repesada (0.01 K)")
        if nombre == "c":
            a.plot(e4["T"], e4["c_intra"], "k--", lw=1, label="c intra-copia")
            a.axvspan(*np.quantile(rw["c_ts"], [0.025, 0.975]), color="red", alpha=0.15, label="IC 95 % de T*")
        a.set_xscale("log"); a.set_title(f"Estado {et}: {nombre}(T)"); a.set_xlabel("T (K)")
        a.set_ylabel("c (k_B por espín)" if nombre == "c" else "χ (1/meV por espín)")
ax[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("figures/E4_bandas.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, (et, e4) in zip(ax, E4.items()):
    ts = e4["rw"]["c_ts"]
    a.hist(ts, bins=40, edgecolor="k")
    a.axvline(np.quantile(ts, 0.025), c="red", ls="--", lw=1)
    a.axvline(np.quantile(ts, 0.975), c="red", ls="--", lw=1, label="IC 95 % de T*")
    a.axvline(e4["meseta"][0], c="gray", ls=":", lw=1)
    a.axvline(e4["meseta"][1], c="gray", ls=":", lw=1, label="meseta del pico")
    a.set_title(f"Estado {et}: distribución bootstrap de T*"); a.set_xlabel("T* (K)"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/E4_Tstar.png", dpi=150); plt.show()


### Alcance de las conclusiones

Un intervalo de $T^*$ solo es confiable si cae en una zona donde la cadena convergió. Esta celda compara el IC de $T^*$ con las temperaturas no convergidas de la sección de convergencia.


In [ ]:
for et in PRINCIPAL:
    conv, T = PT[et]["conv"], PT[et]["T"]
    malas = T[~conv]
    if malas.size:
        cortes = np.where(np.diff(np.searchsorted(T, malas)) > 1)[0]
        grupos = np.split(malas, cortes + 1)
        rangos = ", ".join(f"[{g[0]:.2f}, {g[-1]:.2f}] K" for g in grupos)
    else:
        rangos = "ninguno"
    ic = np.quantile(E4[et]["rw"]["c_ts"], [0.025, 0.975])
    dentro = bool(np.any(~conv & (T >= ic[0] - 0.5) & (T <= ic[1] + 0.5)))
    print(f"Estado {et}: intervalos de T sin convergencia: {rangos} | T* IC95 [{ic[0]:.2f}, {ic[1]:.2f}] K -> "
          f"{'PROVISIONAL (cae junto a una zona no convergida)' if dentro else 'en zona convergida'}")

### Guía de lectura: bandas de $c$ y $\chi$, temperatura del pico y meseta

Tablas y figuras: `E4.md`, `E4_chi.md`, `E4_intra.md`, `E4_bandas.png` y `E4_Tstar.png`.

**Qué se hizo y qué representa.** Se estimaron el calor específico por espín $c(T)$ y la susceptibilidad $\chi(T)$ con su incertidumbre. Las copias independientes se remuestrearon con reemplazo ($B=2000$) con índices generados por inversión discreta, y se construyeron bandas puntuales del 95 % con dos métodos: percentil y BCa. Las curvas se evaluaron en una rejilla de 0.01 K por repesado de histogramas, con control del tamaño efectivo de Kish. La temperatura característica $T^*$ se definió **solo con $c(T)$**: es el argmax de $c$ en la ventana 6–20 K, calculado en cada remuestra, lo que produce su distribución. `E4.md` reporta, por estado, el pico en la escalera (ventana), $T^*$ repesada, la mediana bootstrap de $T^*$, su error estándar bootstrap, el IC95 percentil, la meseta del pico $[T_{lo},T_{hi}]$ y el valor de $c$ en el pico. `E4_chi.md` indica el argmax de $\chi$ en la ventana y si cae en un borde. `E4_intra.md` compara $c(T^*)$ con el estimador agrupado y con el intra-copia. `E4_bandas.png` muestra las bandas con las temperaturas no convergidas sombreadas en gris y $c$ intra-copia punteada, y `E4_Tstar.png` la distribución bootstrap de $T^*$.

**Por qué es la prueba adecuada.** Las muestras dentro de una cadena están correlacionadas y el bootstrap clásico no es válido allí; las copias, en cambio, son independientes y constituyen la unidad natural de remuestreo. Como $c$ y $\chi$ son varianzas, su distribución de muestreo es asimétrica, y el intervalo BCa corrige sesgo y asimetría; comparar BCa con el percentil indica cuánto importan esas correcciones. El repesado permite buscar el máximo sobre una rejilla más fina que la escalera.

**La meseta del pico y por qué el argmax es inestable.** En un disco finito el pico de $c$ es redondeado y plano. Cerca del máximo $T_0$, $c(T)\approx c_0-\tfrac12\kappa(T-T_0)^2$ con curvatura $\kappa$ pequeña. Si el ruido estadístico en $c$ tiene escala $\sigma_c$, dos temperaturas $T$ y $T_0$ son indistinguibles cuando $\tfrac12\kappa(T-T_0)^2\lesssim\sigma_c$, es decir, cuando $|T-T_0|\lesssim\sqrt{2\sigma_c/\kappa}$. La incertidumbre del argmax crece entonces como $\sqrt{\sigma_c/\kappa}$: **el error de decisión es proporcional a la planitud** ($1/\sqrt\kappa$). Cuanto más plana la curva, más ancho es el conjunto de temperaturas que podrían ser el máximo y más inestable es la ubicación de un solo punto. Por eso se reportan, además del argmax, la mediana bootstrap, el IC percentil y la **meseta** $[T_{lo},T_{hi}]$ (el intervalo de temperaturas cuyo $c$ es compatible con el del pico). El ancho de la meseta es la medida honesta de qué tan bien queda determinada $T^*$.

**Estimador agrupado frente a intra-copia.** El estimador agrupado (pooled) calcula $\operatorname{Var}(H)$ sobre todas las muestras mezcladas de las 16 copias; su varianza incluye la **varianza entre copias**, es decir, las diferencias entre las medias de copias distintas. Si todas las copias están en equilibrio, esa componente es pequeña. Si cada copia está atascada en un mínimo diferente, la varianza entre copias domina y el estimador agrupado sobrestima $c$, lo que genera picos espurios en la zona fría (en una corrida anterior apareció uno a 0.5–0.6 K, que se identificó como artefacto). El estimador intra-copia promedia las varianzas calculadas dentro de cada copia y descarta esa componente. La comparación en `E4_intra.md` (ventana y 0.5–2 K) muestra cuánto de $c$ proviene de la mezcla de copias atascadas. Coincidencia en la ventana respalda el equilibrio; discrepancia en 0.5–2 K confirma que esa zona no es interpretable.

**Por qué $\chi$ no define $T^*$.** Se espera que la susceptibilidad $\chi(T)$ decrezca de forma monótona en la ventana 6–20 K, sin pico interior. En ese caso su argmax cae en un borde y coincide con el límite de la ventana, no con una propiedad del sistema; `E4_chi.md` lo señala como «sin pico interior». Definir $T^*$ con $\chi$ confundiría una convención de la ventana con una temperatura característica. La curva de $\chi$ se muestra, pero no se usa para $T^*$.

**Cómo leer el resultado.** Una banda estrecha indica que las copias coinciden en el valor de $c$ o $\chi$ a esa temperatura; una banda ancha, desacuerdo entre copias, esperable cerca del pico. Una distribución de $T^*$ ancha refleja la planitud del pico y no un fallo del método. Si la distribución se pega a uno de los bordes de la ventana, el pico real está fuera de ella. Si coinciden percentil y BCa, la distribución bootstrap es aproximadamente simétrica. Las temperaturas sombreadas en gris son las no convergidas; lo calculado allí no se interpreta. Los intervalos cuantifican variabilidad entre copias, no sesgo por falta de equilibrio, ni el efecto del tamaño del disco, ni el número limitado de copias.


## Discusión

Se redacta después de interpretar los resultados de esta corrida. Las tablas y figuras se regeneran con los datos nuevos; ninguna cifra de versiones anteriores se conserva.


In [ ]:
subprocess.run("zip -qr entrega1_v3_tablas_figuras.zip tablas figures", shell=True, check=True)
print(f"tiempo total del cuaderno: {(time.time() - t0_global) / 3600:.2f} h")
